# Algerian KYC OCR — complete pipeline (Qwen3.6, local)

Self-contained. No dependency on any previous notebook. Edit **section 05 (Configuration)**, run
top to bottom.

---

## Why the parallel Qwen3.8 run returned `!!!!!!`

This is almost certainly **not** a prompting failure. It has a specific numerical signature:

```
logits become NaN  ->  argmax() over an all-NaN row returns index 0
                   ->  the model emits token id 0 at every step
                   ->  id 0 decodes to "!" in Qwen/LLaMA-family byte-level BPE
                   ->  output is "!!!!!!..." for exactly max_new_tokens steps
```

Three details in your report support this over a bad prompt: it repeated **consistently** (a weak
prompt produces varied junk, NaN does not), the output was **punctuation only**, and it appeared on
**full-page images**, which is where the vision tower does the most numerically aggressive work.

Ranked causes, all checked at runtime by section 16:

| # | Cause | Why it produces NaN |
|---|---|---|
| 1 | `torch_dtype=float16` on a bf16 checkpoint | Qwen-VL vision towers overflow fp16's 65504 ceiling; bf16 has fp32's exponent range |
| 2 | FP8 applied to the vision tower / patch merger | per-block FP8 scales can underflow to zero → division by zero |
| 3 | attention backend mismatch with padding | a fully-masked attention row gives `softmax(-inf)` = NaN |
| 4 | processor/model mismatch | image placeholders not expanded → vision features misaligned |

**The four cheap, decisive tests this notebook runs before touching a single customer document:**

- **A.** `tokenizer.decode([0]) == "!"` — confirms the id-0 signature exists in this vocabulary
- **B.** are the generated ids *all zero*? — proves argmax-on-NaN rather than real text
- **C.** `torch.isnan(logits).any()` on a real forward pass — catches it at the source
- **D.** NaN sweep over model parameters — catches a broken load or quantisation

If any fire, the notebook prints `MODEL_OCR_SANITY_CHECK = FAILED` with the probable cause and
**refuses to start the pipeline**. That is the difference between finding this in 90 seconds and
finding it after an hour of processing.

---

## Architecture: why the call count is now the page count

Previous versions issued a separate Qwen call per field, per title, per MRZ, per handwriting
region — 7 calls taking 15 minutes. This version inverts that:

```
page image ──► ONE Qwen call ──► raw transcription of the whole page
                                          │
                                          ▼
                              Python parses the transcription
                              (labels, regex, MRZ pattern)
                                          │
                        identity fields · names · MRZ · titles
```

Structured extraction is **CPU work on text the model already produced**. It reads the same
pixels once. A targeted second call happens only in one situation: an MRZ-shaped region was
detected on the page but the transcription did not contain readable MRZ lines.

**Planned calls = pages + (0–1 MRZ recovery per identity document).** The planner prints this
before inference and aborts if it exceeds the configured ceiling.

## How to run

1. Leave `DRY_RUN = True`. Confirm the discovered customers, PDFs, real page counts and planned
   call count.
2. Set `DRY_RUN = False` with `BENCHMARK_MODE = True` to process one customer and read the
   timings.
3. Only then raise `BENCHMARK_CUSTOMERS` or set `BENCHMARK_MODE = False`.

In [ ]:
# =========================================================================
# 03 — INSTALLATION
# =========================================================================
# Domino images normally ship torch/transformers/accelerate already. Reinstalling torch can break
# the CUDA build, so nothing is installed unless INSTALL_MISSING is set explicitly.
#
# Pinned set known to work together for Qwen-VL family multimodal inference on an H100:
#   torch>=2.3,<2.8            SDPA with the memory-efficient kernel
#   transformers>=4.49,<5      AutoModelForImageTextToText + FineGrainedFP8Config
#   accelerate>=0.30           device_map / max_memory
#   tokenizers>=0.19  safetensors>=0.4.3
#   pillow>=10.0  pymupdf>=1.24  numpy>=1.24  pandas>=2.0
#   opencv-python-headless>=4.9   (MRZ localisation only; optional)
#
# NEVER installed, NEVER imported: flash_attn.
# flash-linear-attention==0.5.2 is already present and is CHECKED (section 13), not assumed.
INSTALL_MISSING = False
PIN = {"transformers": ">=4.49,<5", "accelerate": ">=0.30", "tokenizers": ">=0.19",
       "safetensors": ">=0.4.3", "pillow": ">=10.0", "pymupdf": ">=1.24",
       "numpy": ">=1.24", "pandas": ">=2.0", "opencv-python-headless": ">=4.9"}

if INSTALL_MISSING:
    import subprocess, sys
    for pkg, spec in PIN.items():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{pkg}{spec}"], check=False)
    print("install attempted -- RESTART THE KERNEL before continuing")
else:
    print("INSTALL_MISSING = False (recommended). Pinned targets:")
    for k, v in PIN.items():
        print(f"  {k}{v}")

In [ ]:
# =========================================================================
# 04 — IMPORTS
# =========================================================================
from __future__ import annotations

import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import io, re, sys, json, time, math, random, zipfile, logging, platform, subprocess, traceback
import difflib, unicodedata, importlib, inspect
from collections import defaultdict, OrderedDict
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone, date
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image

Image.MAX_IMAGE_PIXELS = None

# PyMuPDF under its current name. The deprecated `fitz` alias is not used anywhere.
try:
    import pymupdf
except Exception as exc:
    pymupdf = None
    print("pymupdf import failed:", exc)

try:
    import cv2
except Exception:
    cv2 = None

try:
    import torch
except Exception:
    torch = None

try:
    import transformers
except Exception:
    transformers = None

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)-7s %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger("kyc")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
utcnow = lambda: datetime.now(timezone.utc).isoformat()
print("run id:", RUN_ID)

In [ ]:
# =========================================================================
# 05 — CONFIGURATION  (single central object; nothing is configured elsewhere)
# =========================================================================
@dataclass
class Config:
    # ---- paths -----------------------------------------------------------
    MODELHUB_ROOT: Path = Path("/domino/edv/modelhub")
    MODEL_PATH: Optional[str] = None          # None = auto-discover (section 09)
    MODEL_NAME_HINT: str = "Qwen3.6"          # substring used to rank candidate directories
    CUSTOMER_ROOT: Path = Path("/domino/datasets/local/kyc/customers")
    ZIP_PATH: Optional[Path] = Path("/domino/datasets/local/kyc/kyc_documents.zip")
    CUSTOMER_DATABASE_PATH: Optional[Path] = Path("/domino/datasets/local/kyc/customers.csv")
    OUTPUT_DIR: Path = Path("/mnt/kyc_output")

    # ---- run control -----------------------------------------------------
    DRY_RUN: bool = True                      # plan only; no Qwen call can execute
    BENCHMARK_MODE: bool = True
    BENCHMARK_CUSTOMERS: int = 1
    RANDOM_SEED: int = 42
    PIN_CUSTOMER: Optional[str] = None        # process one specific folder name
    RESUME: bool = False                      # off: RESUME once silently skipped a whole run

    # ---- rendering / images ----------------------------------------------
    RENDER_DPI: int = 200                     # render once at this DPI, keep it
    MAX_IMAGE_DIMENSION: int = 1600
    MAX_VISUAL_TOKENS: int = 2048             # processor max_pixels; THE prefill cost knob
    MIN_VISUAL_TOKENS: int = 256
    MRZ_UPSCALE: float = 3.0
    HANDWRITING_UPSCALE: float = 2.5
    REGION_RENDER_DPI: int = 500              # ROI re-render from the PDF, not pixel upscaling
    USE_CV_PREPROCESSING: bool = True         # cheap CV for MRZ localisation only
    MAX_PAGES_PER_PDF: int = 40               # runaway guard; real counts come from the PDF
    SAVE_DEBUG_IMAGES: bool = False

    # ---- generation ------------------------------------------------------
    MAX_NEW_TOKENS: int = 768                 # a full page transcription; measured, not guessed
    MAX_NEW_TOKENS_MRZ: int = 160
    MAX_NEW_TOKENS_TEST: int = 64
    DO_SAMPLE: bool = False
    TEMPERATURE: float = 0.0
    REPETITION_PENALTY: float = 1.0           # MUST be 1.0: >1 corrupts repeats and "<<<"
    NO_REPEAT_NGRAM_SIZE: int = 0             # MUST be 0: would forbid "<<" in an MRZ
    DISABLE_THINKING: bool = True

    # ---- model -----------------------------------------------------------
    TORCH_DTYPE: str = "bfloat16"             # NOT float16: see the NaN analysis in section 01
    DEVICE_MAP: str = "auto"
    ATTENTION_BACKEND: str = "sdpa"           # flash_attn is never imported
    USE_FP8: bool = True                      # only applied if the model supports it (section 11)
    FP8_SKIP_MODULES: Tuple[str, ...] = ("lm_head", "visual", "vision_tower", "vision_model",
                                         "merger", "multi_modal_projector")
    RESERVE_VRAM_GIB: float = 6.0
    ALLOW_CPU_OFFLOAD: bool = False           # False: fail loudly rather than run 50x slower

    # ---- limits ----------------------------------------------------------
    MAX_QWEN_CALLS_PER_PDF: int = 8
    MAX_QWEN_CALLS_PER_CUSTOMER: int = 30
    MAX_TOTAL_QWEN_CALLS: int = 5000
    MAX_RUNTIME_SECONDS: float = 3600.0
    MAX_QWEN_CALL_SECONDS: float = 180.0
    WARN_QWEN_CALL_SECONDS: float = 45.0
    MAX_TARGETED_RETRIES: int = 1

    # ---- verification ----------------------------------------------------
    DB_COL_ID: str = "Id tiers"
    DB_COL_NAME: str = "Nom abrégé tiers"
    DB_COL_DOB: str = "Date de naissance"
    DB_COL_EXPIRY: str = "Date d'expiration du Document"
    NAME_FUZZY_THRESHOLD: float = 0.90

    MOCK_MODEL: bool = False                  # rehearse the whole pipeline without a GPU

    def dirs(self) -> Dict[str, Path]:
        d = {k: self.OUTPUT_DIR / v for k, v in
             {"raw": "01_raw_ocr", "results": "02_results", "reports": "03_reports",
              "debug": "04_debug", "logs": "05_logs"}.items()}
        for p in d.values():
            p.mkdir(parents=True, exist_ok=True)
        return d


CFG = Config()
DIRS = CFG.dirs()

IDENTITY_DOC, DOMICILE_DOC = "JUSTIFICATIF IDENTITE.PDF", "JUSTIFICATIF DOMICILE.PDF"
CONVENTION_DOC, FATCA_DOC = "CONVENTION COMPTE.PDF", "FATCA.PDF"
SIGNATURE_DOC = "CARTON SIGNATURE.PDF"
DOCUMENTS_REQUIRED = [IDENTITY_DOC, DOMICILE_DOC, CONVENTION_DOC, FATCA_DOC, SIGNATURE_DOC]
PRESENCE_COLUMNS = OrderedDict([(IDENTITY_DOC, "identity_document_exists"),
                                (DOMICILE_DOC, "domicile_document_exists"),
                                (CONVENTION_DOC, "convention_compte_exists"),
                                (FATCA_DOC, "fatca_exists"),
                                (SIGNATURE_DOC, "signature_card_exists")])
IDENTITY_FIELDS = ["surname_latin", "given_names_latin", "date_of_birth", "place_of_birth",
                   "nationality", "sex", "document_number", "issue_date", "expiry_date",
                   "issuing_authority", "personal_number", "mrz"]
NAME_FIELDS = ["surname_latin", "given_names_latin"]

print(f"{'output':<22}: {CFG.OUTPUT_DIR}")
print(f"{'DRY_RUN':<22}: {CFG.DRY_RUN}")
print(f"{'BENCHMARK_MODE':<22}: {CFG.BENCHMARK_MODE} ({CFG.BENCHMARK_CUSTOMERS} customer(s))")
print(f"{'render / tokens':<22}: {CFG.RENDER_DPI} dpi | {CFG.MAX_VISUAL_TOKENS} visual tokens "
      f"| {CFG.MAX_NEW_TOKENS} max_new_tokens")
print(f"{'dtype / attention':<22}: {CFG.TORCH_DTYPE} / {CFG.ATTENTION_BACKEND}")

In [ ]:
# =========================================================================
# 02 / 07 / 08 — ENVIRONMENT, GPU AND CUDA DIAGNOSTICS
# =========================================================================
def versions() -> Dict[str, str]:
    out = {"python": platform.python_version(), "platform": platform.platform()}
    for name in ("torch", "torchvision", "transformers", "accelerate", "tokenizers",
                 "safetensors", "numpy", "pandas", "PIL", "pymupdf", "cv2", "fla"):
        try:
            m = importlib.import_module(name)
            out[name] = getattr(m, "__version__", "present")
        except Exception:
            out[name] = "NOT INSTALLED"
    if torch is not None:
        out["cuda (torch build)"] = str(torch.version.cuda)
        out["cuda available"] = str(torch.cuda.is_available())
    return out


for k, v in versions().items():
    print(f"{k:<22}: {v}")

try:
    print(f"{'flash_attn present':<22}: "
          f"{importlib.util.find_spec('flash_attn') is not None} (never imported or used)")
except Exception:
    pass


def gpu_snapshot(tag: str) -> Dict[str, Any]:
    """One row of memory truth. torch.memory_allocated()==0 BEFORE the model loads is normal --
    it measures this process's allocations, not the card. nvidia-smi shows the card."""
    rec: Dict[str, Any] = {"tag": tag, "timestamp": utcnow()}
    if torch is not None and torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        rec.update({
            "torch_allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 2),
            "torch_reserved_gib": round(torch.cuda.memory_reserved() / 2**30, 2),
            "torch_peak_allocated_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2),
            "torch_peak_reserved_gib": round(torch.cuda.max_memory_reserved() / 2**30, 2),
            "cuda_free_gib": round(free / 2**30, 2),
            "cuda_total_gib": round(total / 2**30, 2)})
        try:
            smi = subprocess.run(
                ["nvidia-smi", "--query-gpu=memory.used,memory.free,utilization.gpu",
                 "--format=csv,noheader,nounits"], capture_output=True, text=True,
                timeout=10).stdout.strip().split("\n")[0].split(",")
            rec.update({"smi_used_mib": int(smi[0]), "smi_free_mib": int(smi[1]),
                        "smi_util_pct": int(smi[2])})
        except Exception:
            pass
    MEMORY_TRACE.append(rec)
    return rec


MEMORY_TRACE: List[Dict[str, Any]] = []

if torch is not None and torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"\ngpu[{i}]                : {p.name}")
        print(f"{'total memory':<22}: {p.total_memory/2**30:.1f} GiB")
        print(f"{'compute capability':<22}: sm_{p.major}{p.minor}")
    snap = gpu_snapshot("before_model_load")
    print(f"\n{'MEMORY BEFORE LOAD':<22}: {json.dumps({k: v for k, v in snap.items() if k not in ('tag','timestamp')})}")
    print("  note: torch_allocated 0.0 GiB here is EXPECTED -- nothing has been loaded yet.")
    print("  cuda_free below total means ANOTHER PROCESS holds memory on this card.")
else:
    print("\nno CUDA device visible -- set CFG.MOCK_MODEL = True to rehearse the pipeline")

In [ ]:
# =========================================================================
# 09 — MODEL DISCOVERY AND CONFIGURATION INSPECTION
# =========================================================================
# "Qwen3.6" is not assumed to exist under any particular name. The ModelHub tree is scanned, every
# candidate's config.json is read, and the architecture is determined from the file rather than
# from the folder name.
def scan_modelhub(root: Path, hint: str = "") -> pd.DataFrame:
    rows = []
    if not Path(root).exists():
        log.warning("ModelHub root does not exist: %s", root)
        return pd.DataFrame()
    for cfg_file in Path(root).rglob("config.json"):
        try:
            j = json.loads(cfg_file.read_text())
        except Exception:
            continue
        d = cfg_file.parent
        archs = j.get("architectures") or []
        rows.append({
            "path": str(d), "name": d.parent.name if d.name == "main" else d.name,
            "model_type": j.get("model_type"), "architectures": ",".join(archs),
            "has_vision": bool(j.get("vision_config") or "vl" in " ".join(archs).lower()),
            "quant": (j.get("quantization_config") or {}).get("quant_method"),
            "torch_dtype": j.get("torch_dtype"),
            "hint_match": hint.lower().replace(".", "").replace("-", "") in
                          str(d).lower().replace(".", "").replace("-", ""),
        })
    df = pd.DataFrame(rows)
    if len(df):
        df = df.sort_values(["hint_match", "has_vision"], ascending=False).reset_index(drop=True)
    return df


CANDIDATES = scan_modelhub(CFG.MODELHUB_ROOT, CFG.MODEL_NAME_HINT)
if len(CANDIDATES):
    print("models found under ModelHub:\n")
    print(CANDIDATES[["name", "model_type", "architectures", "has_vision", "quant",
                      "torch_dtype"]].to_string(index=False))
else:
    print("no config.json found under", CFG.MODELHUB_ROOT)


def choose_model_path(cfg: Config = CFG) -> str:
    if cfg.MODEL_PATH:
        return cfg.MODEL_PATH
    if not len(CANDIDATES):
        if cfg.MOCK_MODEL:
            log.warning("no model found under %s, but MOCK_MODEL is on: the pipeline can still "
                        "be rehearsed end to end without a GPU", cfg.MODELHUB_ROOT)
            return "<MOCK: no model directory>"
        raise FileNotFoundError(
            f"No model found under {cfg.MODELHUB_ROOT}. Set CFG.MODEL_PATH explicitly, or set "
            "CFG.MOCK_MODEL = True to rehearse the pipeline without the model.")
    vis = CANDIDATES[CANDIDATES["has_vision"]]
    hinted = vis[vis["hint_match"]] if len(vis) else CANDIDATES[CANDIDATES["hint_match"]]
    pick = (hinted.iloc[0] if len(hinted) else (vis.iloc[0] if len(vis) else CANDIDATES.iloc[0]))
    if not pick["hint_match"]:
        log.warning("no directory matches the hint %r; falling back to %s. Set CFG.MODEL_PATH "
                    "if this is the wrong model.", cfg.MODEL_NAME_HINT, pick["path"])
    if not pick["has_vision"]:
        log.error("selected model has NO vision tower -- it cannot read images: %s", pick["path"])
    return pick["path"]


MODEL_PATH = choose_model_path(CFG)
MODEL_CONFIG = (json.loads((Path(MODEL_PATH) / "config.json").read_text())
                if not MODEL_PATH.startswith("<MOCK")
                and (Path(MODEL_PATH) / "config.json").exists() else {})
print("\n" + "=" * 60)
print("SELECTED MODEL")
print("=" * 60)
print(f"{'path':<24}: {MODEL_PATH}")
for k in ("model_type", "architectures", "torch_dtype", "quantization_config"):
    print(f"{k:<24}: {MODEL_CONFIG.get(k)}")
print(f"{'has vision_config':<24}: {'vision_config' in MODEL_CONFIG}")
vc = MODEL_CONFIG.get("vision_config") or {}
if vc:
    print(f"{'vision hidden/patch':<24}: {vc.get('hidden_size')} / {vc.get('patch_size')}")
tpl = ([p.name for p in Path(MODEL_PATH).glob("*template*")] +
       [p.name for p in Path(MODEL_PATH).glob("*processor*")] +
       [p.name for p in Path(MODEL_PATH).glob("tokenizer*")]) \
    if not MODEL_PATH.startswith("<MOCK") else []
print(f"{'processor/template files':<24}: {sorted(set(tpl))[:6]}")

In [ ]:
# =========================================================================
# 11 / 13 — QUANTIZATION DECISION AND ATTENTION BACKEND
# =========================================================================
# FP8 is applied ONLY if this specific checkpoint warrants it. If the checkpoint is already
# quantised, its own configuration is used and nothing is converted by hand -- calling .half()
# or .float() on a quantised model is one of the documented routes to NaN logits.
FP8_REPORT = {"requested": CFG.USE_FP8, "class_found": False, "import_path": None,
              "applied": False, "reason": ""}
FP8_QUANT_CONFIG = None

if CFG.USE_FP8 and not CFG.MOCK_MODEL:
    ckpt_quant = (MODEL_CONFIG.get("quantization_config") or {})
    if ckpt_quant:
        FP8_REPORT["reason"] = (f"checkpoint already declares quant_method="
                                f"{ckpt_quant.get('quant_method')}; using it as stored, no manual "
                                f"conversion")
    else:
        # Verified against the installed package: the class is NOT in
        # transformers.integrations.finegrained_fp8 (that module holds FP8Linear and the
        # quantiser). It is defined in transformers.utils.quantization_config and re-exported at
        # the top level. The documented path is tried first regardless.
        for mod_name, attr in [("transformers.integrations.finegrained_fp8",
                                "FineGrainedFP8Config"),
                               ("transformers", "FineGrainedFP8Config"),
                               ("transformers.utils.quantization_config",
                                "FineGrainedFP8Config")]:
            try:
                cls = getattr(importlib.import_module(mod_name), attr)
                params = [p for p in inspect.signature(cls.__init__).parameters
                          if p not in ("self", "args", "kwargs")]
                kwargs = {}
                if "activation_scheme" in params:
                    kwargs["activation_scheme"] = "dynamic"
                if "weight_block_size" in params:
                    kwargs["weight_block_size"] = (128, 128)
                if "modules_to_not_convert" in params:
                    # The vision tower stays in bf16. Per-block FP8 scales on the patch embedding
                    # can underflow to zero -> division by zero -> NaN logits -> "!!!!!!".
                    kwargs["modules_to_not_convert"] = list(CFG.FP8_SKIP_MODULES)
                FP8_QUANT_CONFIG = cls(**kwargs)
                FP8_REPORT.update({"class_found": True, "import_path": f"{mod_name}.{attr}",
                                   "params": params, "configured": kwargs,
                                   "reason": "constructed for an unquantised checkpoint"})
                break
            except Exception as exc:
                FP8_REPORT["reason"] = f"{type(exc).__name__}"
else:
    FP8_REPORT["reason"] = "USE_FP8 False or MOCK_MODEL True"

print("FP8 decision:", json.dumps(FP8_REPORT, default=str, indent=1))

# ---- flash-linear-attention: checked, never assumed ----
FLA_REPORT = {"installed": False, "version": None, "compatible": False, "used": False,
              "reason": ""}
try:
    import fla
    FLA_REPORT.update({"installed": True, "version": getattr(fla, "__version__", "unknown")})
    FLA_REPORT["reason"] = ("flash-linear-attention provides LINEAR-attention kernels (GLA, "
                            "RetNet, RWKV, DeltaNet). This checkpoint is a standard "
                            "softmax-attention transformer, so the kernels compute a different "
                            "function and are not a drop-in substitute. transformers' "
                            "attn_implementation also does not accept it.")
except Exception as exc:
    FLA_REPORT["reason"] = f"not importable ({type(exc).__name__})"


def resolve_attention(cfg: Config = CFG) -> str:
    req = (cfg.ATTENTION_BACKEND or "sdpa").lower()
    if req in ("flash_attention_2", "flash_attn"):
        log.warning("FlashAttention is not a dependency of this notebook; using sdpa")
        req = "sdpa"
    if req in ("fla", "flash_linear_attention") and not FLA_REPORT["compatible"]:
        log.warning("flash-linear-attention not compatible: %s", FLA_REPORT["reason"])
        req = "sdpa"
    if req == "sdpa" and torch is not None and \
            not hasattr(torch.nn.functional, "scaled_dot_product_attention"):
        req = "eager"
    return req


ATTENTION_IMPL = resolve_attention(CFG)
print(f"\n{'flash-linear-attention':<26}: installed={FLA_REPORT['installed']} "
      f"version={FLA_REPORT['version']}")
print(f"{'FLA compatible / used':<26}: {FLA_REPORT['compatible']} / {FLA_REPORT['used']}")
print(f"{'FLA reason':<26}: {FLA_REPORT['reason'][:110]}")
print(f"{'ACTUAL attention backend':<26}: {ATTENTION_IMPL}")

In [ ]:
# =========================================================================
# 10 / 11 / 12 / 14 — PROCESSOR AND MODEL (one instance each) + DEVICE AUDIT
# =========================================================================
class MockEngine:
    is_mock, name, attn = True, "MOCK", "n/a"
    device_map_summary, placement, quantization = {"mock": 1}, "MOCK", {}
    load_seconds = processor_seconds = 0.0
    model_class = "MockEngine"


class QwenEngine:
    _instance = None
    is_mock = False

    def __init__(self, cfg: Config = CFG):
        from transformers import AutoConfig, AutoProcessor
        import transformers as tf
        assert torch is not None, "PyTorch required"
        self.cfg = cfg
        self.hf_config = AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=True,
                                                    local_files_only=True)
        archs = list(getattr(self.hf_config, "architectures", []) or [])

        # ---- processor: ONE instance. max_pixels bounds the prefill cost. ----
        t0 = time.perf_counter()
        try:
            self.processor = AutoProcessor.from_pretrained(
                MODEL_PATH, trust_remote_code=True, local_files_only=True,
                min_pixels=cfg.MIN_VISUAL_TOKENS * 28 * 28,
                max_pixels=cfg.MAX_VISUAL_TOKENS * 28 * 28)
            self.pixel_cap = True
        except TypeError:
            self.processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True,
                                                           local_files_only=True)
            self.pixel_cap = False
            log.warning("processor does not accept min/max_pixels; visual tokens are uncapped. "
                        "Lower CFG.RENDER_DPI instead.")
        self.processor_seconds = round(time.perf_counter() - t0, 2)
        self.tokenizer = getattr(self.processor, "tokenizer", self.processor)
        self.has_chat_template = bool(getattr(self.processor, "chat_template", None) or
                                      getattr(self.tokenizer, "chat_template", None))

        kwargs: Dict[str, Any] = dict(
            torch_dtype=getattr(torch, cfg.TORCH_DTYPE),    # bf16: see the NaN analysis
            device_map=cfg.DEVICE_MAP, trust_remote_code=True, local_files_only=True,
            low_cpu_mem_usage=True, attn_implementation=ATTENTION_IMPL)
        if FP8_QUANT_CONFIG is not None:
            kwargs["quantization_config"] = FP8_QUANT_CONFIG
        if cfg.RESERVE_VRAM_GIB > 0 and torch.cuda.is_available() and cfg.DEVICE_MAP == "auto":
            mm = {}
            for i in range(torch.cuda.device_count()):
                _, total = torch.cuda.mem_get_info(i)
                mm[i] = f"{max(1, int(total / 2**30) - int(cfg.RESERVE_VRAM_GIB))}GiB"
            if cfg.ALLOW_CPU_OFFLOAD:
                mm["cpu"] = "64GiB"
            kwargs["max_memory"] = mm

        t0, last = time.perf_counter(), None
        self.model = None
        for cls_name in archs + ["AutoModelForImageTextToText", "AutoModelForVision2Seq",
                                 "AutoModelForCausalLM"]:
            if not hasattr(tf, cls_name):
                continue
            try:
                self.model = getattr(tf, cls_name).from_pretrained(MODEL_PATH, **kwargs)
                self.model_class = cls_name
                break
            except Exception as exc:
                last = f"{cls_name}: {type(exc).__name__}: {exc}"
                msg = str(exc).lower()
                if "quantization_config" in kwargs and "quantiz" in msg:
                    log.warning("checkpoint refused an explicit FP8 config; using its own")
                    kwargs.pop("quantization_config")
                elif "attention" in msg and kwargs.get("attn_implementation") != "eager":
                    log.warning("attn_implementation %s refused; using eager",
                                kwargs["attn_implementation"])
                    kwargs["attn_implementation"] = "eager"
        if self.model is None:
            raise RuntimeError(f"could not load the model. last error: {last}")
        self.load_seconds = round(time.perf_counter() - t0, 1)
        self.attn = kwargs.get("attn_implementation", ATTENTION_IMPL)
        self.model.eval()                                   # never training mode
        gc_ = getattr(self.model, "generation_config", None)
        if gc_ is not None:
            gc_.do_sample = cfg.DO_SAMPLE
            gc_.temperature = gc_.top_p = gc_.top_k = None

        # ---- device audit: evidence, not assumption ----
        dm = getattr(self.model, "hf_device_map", None) or {}
        summary = defaultdict(int)
        for _, dev in dm.items():
            summary[str(dev)] += 1
        self.device_map_summary = dict(summary)
        devs = {str(p.device) for _, p in self.model.named_parameters()}
        self.param_devices = sorted(devs)
        has_cuda = any("cuda" in d for d in devs)
        has_cpu = any(d == "cpu" for d in devs)
        self.placement = ("MIXED_CPU_GPU" if has_cuda and has_cpu else
                          "GPU_ONLY" if has_cuda else "CPU_ONLY" if has_cpu else "UNKNOWN")
        try:
            self.first_param_device = str(next(self.model.parameters()).device)
        except Exception:
            self.first_param_device = "?"
        self.vision_device = next((str(p.device) for n, p in self.model.named_parameters()
                                   if any(h in n for h in ("visual", "vision"))), "n/a")
        qc = getattr(self.model.config, "quantization_config", None)
        method = getattr(qc, "quant_method", None) if qc is not None else None
        n_fp8 = sum(1 for _, m in self.model.named_modules() if type(m).__name__ == "FP8Linear")
        dtypes = defaultdict(int)
        for _, p in list(self.model.named_parameters())[:400]:
            dtypes[str(p.dtype)] += 1
        self.quantization = {"quant_method": str(getattr(method, "value", method)),
                             "n_fp8_linear": n_fp8, "param_dtypes": dict(dtypes)}
        self.name = f"{Path(MODEL_PATH).parent.name} [{self.model_class}]"

    @classmethod
    def get(cls, cfg: Config = CFG):
        if cls._instance is None:
            cls._instance = cls(cfg)
        return cls._instance


def load_engine(cfg: Config = CFG):
    """Model AND processor, exactly once per kernel."""
    return MockEngine() if cfg.MOCK_MODEL else QwenEngine.get(cfg)


gpu_snapshot("before_model_load_2")
ENGINE = load_engine(CFG)
snap_after = gpu_snapshot("after_model_load")

print("=" * 62)
print("MODEL LOADED")
print("=" * 62)
print(f"{'model class':<24}: {getattr(ENGINE, 'model_class', '?')}")
print(f"{'processor':<24}: {type(getattr(ENGINE, 'processor', None)).__name__}")
print(f"{'chat template':<24}: {getattr(ENGINE, 'has_chat_template', '?')}")
print(f"{'visual token cap':<24}: {getattr(ENGINE, 'pixel_cap', '?')}")
print(f"{'attention (actual)':<24}: {ENGINE.attn}")
print(f"{'quantization':<24}: {json.dumps(getattr(ENGINE, 'quantization', {}), default=str)}")
print(f"{'device map':<24}: {getattr(ENGINE, 'device_map_summary', {})}")
print(f"{'param devices':<24}: {getattr(ENGINE, 'param_devices', [])}")
print(f"{'first param device':<24}: {getattr(ENGINE, 'first_param_device', '?')}")
print(f"{'vision module device':<24}: {getattr(ENGINE, 'vision_device', '?')}")
print(f"{'PLACEMENT':<24}: {getattr(ENGINE, 'placement', '?')}")
print(f"{'load time':<24}: model {ENGINE.load_seconds}s | processor {ENGINE.processor_seconds}s")
print(f"\nMEMORY AFTER LOAD       : "
      f"{json.dumps({k: v for k, v in snap_after.items() if k not in ('tag','timestamp')})}")
if getattr(ENGINE, "placement", "") == "MIXED_CPU_GPU":
    print("\n" + "!" * 70)
    print("!! PART OF THE MODEL IS ON CPU. Every forward pass streams weights over PCIe;")
    print("!! this alone turns a ~1 s decode into 60+ s and explains 60-90 s per call.")
    print("!! Fix: lower CFG.RESERVE_VRAM_GIB, free the card, or keep ALLOW_CPU_OFFLOAD False.")
    print("!" * 70)

In [ ]:
# =========================================================================
# 15 / 16 — CUDA INFERENCE TEST AND OCR SANITY GATE
# =========================================================================
# This is the gate that would have caught "!!!!!!" in 90 seconds instead of an hour.
SANITY: Dict[str, Any] = {"passed": None, "tests": {}, "diagnosis": []}


def _degenerate(text: str, ids: Optional[List[int]] = None) -> Optional[str]:
    """Classify a degenerate generation. Returns a reason, or None if the output looks real."""
    t = (text or "").strip()
    if not t:
        return "EMPTY_OUTPUT"
    if ids is not None and len(ids) > 4 and len(set(ids)) == 1:
        return f"SINGLE_TOKEN_REPEATED (id={ids[0]}) -- argmax over NaN logits returns index 0"
    stripped = re.sub(r"\s", "", t)
    if stripped and len(set(stripped)) <= 2 and not re.search(r"[A-Za-z0-9\u0600-\u06FF]", stripped):
        return f"PUNCTUATION_ONLY ({stripped[:12]!r})"
    if len(t) > 40 and len(set(t.split())) <= 2:
        return "REPETITIVE_OUTPUT"
    return None


def raw_generate(image: Image.Image, system: str, user: str, max_new_tokens: int,
                 cfg: Config = CFG, engine=None, return_ids: bool = False) -> Dict[str, Any]:
    """One multimodal generation. Used by both the sanity tests and the pipeline."""
    engine = engine or ENGINE
    rec: Dict[str, Any] = {"text": "", "output_tokens": 0, "input_tokens": None,
                           "visual_tokens": None, "generation_time_s": 0.0,
                           "processor_time_s": 0.0, "tokens_per_s": None,
                           "truncated": False, "status": "OK", "error": None, "ids": []}
    if getattr(engine, "is_mock", False):
        rec["status"] = "MOCK"
        return rec
    messages = [{"role": "system", "content": [{"type": "text", "text": system}]},
                {"role": "user", "content": [{"type": "image"},
                                             {"type": "text", "text": user}]}]
    try:
        text = engine.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=not cfg.DISABLE_THINKING)
    except TypeError:
        text = engine.processor.apply_chat_template(messages, tokenize=False,
                                                    add_generation_prompt=True)
    try:
        t0 = time.perf_counter()
        inputs = engine.processor(text=[text], images=[image], return_tensors="pt")
        inputs = {k: (v.to(engine.model.device) if hasattr(v, "to") else v)
                  for k, v in inputs.items()}
        rec["processor_time_s"] = round(time.perf_counter() - t0, 3)
        rec["input_tokens"] = int(inputs["input_ids"].shape[1])
        grid = inputs.get("image_grid_thw")
        if grid is not None:
            try:
                rec["visual_tokens"] = int(grid.prod(dim=-1).sum().item() // 4)
            except Exception:
                pass
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.inference_mode():
            out = engine.model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=cfg.DO_SAMPLE, num_beams=1,
                repetition_penalty=cfg.REPETITION_PENALTY,
                no_repeat_ngram_size=cfg.NO_REPEAT_NGRAM_SIZE,
                pad_token_id=getattr(engine.tokenizer, "pad_token_id", None)
                             or getattr(engine.tokenizer, "eos_token_id", None))
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        rec["generation_time_s"] = round(time.perf_counter() - t0, 3)
        seq = out[0][inputs["input_ids"].shape[1]:]
        ids = [int(x) for x in seq]
        rec["ids"] = ids if return_ids else ids[:16]
        rec["output_tokens"] = len(ids)
        rec["truncated"] = len(ids) >= max_new_tokens
        # RAW: exactly what the model produced. No strip, no normalisation.
        rec["text"] = engine.processor.decode(seq, skip_special_tokens=True,
                                              clean_up_tokenization_spaces=False)
        if rec["generation_time_s"] > 0:
            rec["tokens_per_s"] = round(len(ids) / rec["generation_time_s"], 1)
        del out, inputs
    except Exception as exc:
        rec["status"] = "ERROR"
        rec["error"] = f"{type(exc).__name__}: {exc}"
    return rec


def run_sanity_gate(cfg: Config = CFG) -> Dict[str, Any]:
    if getattr(ENGINE, "is_mock", False):
        SANITY.update({"passed": True, "tests": {"mock": "skipped"}})
        print("MOCK engine: sanity gate skipped")
        return SANITY

    # ---- TEST A: does token id 0 decode to '!' in this vocabulary? ----
    try:
        tok0 = ENGINE.tokenizer.decode([0])
        SANITY["tests"]["token_id_0_decodes_to"] = repr(tok0)
        print(f"A. tokenizer.decode([0]) = {tok0!r}"
              f"{'   <- the !!!!!! signature exists in this vocab' if tok0.strip() == '!' else ''}")
    except Exception as exc:
        SANITY["tests"]["token_id_0_decodes_to"] = f"error: {exc}"

    # ---- TEST D: NaN anywhere in the loaded weights? ----
    nan_params = []
    for n, p in ENGINE.model.named_parameters():
        try:
            if torch.isnan(p).any() or torch.isinf(p).any():
                nan_params.append(n)
                if len(nan_params) >= 5:
                    break
        except Exception:
            pass
    SANITY["tests"]["nan_parameters"] = nan_params
    print(f"D. NaN/Inf parameters   : {nan_params if nan_params else 'none'}")

    # ---- TEST 1: simple image understanding on a synthetic image ----
    img = Image.new("RGB", (640, 320), "white")
    try:
        from PIL import ImageDraw
        d = ImageDraw.Draw(img)
        d.rectangle([40, 40, 600, 280], outline="black", width=4)
        d.text((80, 140), "NOM: TEST 12345", fill="black")
    except Exception:
        pass
    gpu_snapshot("before_cuda_test")
    r1 = raw_generate(img, "You describe images literally.",
                      "What text is written in this image? Answer in one short line.",
                      cfg.MAX_NEW_TOKENS_TEST, cfg, return_ids=True)
    snap = gpu_snapshot("after_cuda_test")
    SANITY["tests"]["test1"] = {k: r1[k] for k in
                                ("text", "output_tokens", "input_tokens", "visual_tokens",
                                 "generation_time_s", "tokens_per_s", "status", "error")}
    print("\n" + "=" * 62)
    print("TEST 1 -- CONTROLLED MULTIMODAL INFERENCE")
    print("=" * 62)
    print(f"input image size   : {img.size}")
    print(f"input tokens       : {r1['input_tokens']}  (visual: {r1['visual_tokens']})")
    print(f"model device       : {getattr(ENGINE, 'first_param_device', '?')} "
          f"({getattr(ENGINE, 'placement', '?')})")
    print(f"generation time    : {r1['generation_time_s']}s")
    print(f"output tokens      : {r1['output_tokens']}")
    print(f"tokens/second      : {r1['tokens_per_s']}")
    print(f"GPU peak allocated : {snap.get('torch_peak_allocated_gib')} GiB")
    print(f"first 16 token ids : {r1['ids'][:16]}")
    print(f"RAW MODEL OUTPUT   : {r1['text'][:400]!r}")

    reason = _degenerate(r1["text"], r1["ids"]) if r1["status"] == "OK" else r1["error"]
    SANITY["tests"]["test1_degenerate"] = reason
    if r1["status"] != "OK" or reason:
        SANITY["passed"] = False
        SANITY["diagnosis"] = [
            f"degeneration: {reason}",
            "1. dtype -- a bf16 checkpoint loaded as float16 overflows in the vision tower. "
            f"Currently CFG.TORCH_DTYPE={cfg.TORCH_DTYPE} (bfloat16 is correct).",
            "2. FP8 on the vision tower -- per-block scales can underflow to zero. "
            f"FP8_SKIP_MODULES={list(cfg.FP8_SKIP_MODULES)}; applied={FP8_REPORT.get('applied')}.",
            f"3. attention backend -- currently {ENGINE.attn}; try CFG.ATTENTION_BACKEND='eager'.",
            "4. processor/model mismatch -- check that input_tokens above grew with the image "
            "(if visual_tokens is None the image placeholders were not expanded).",
            f"5. placement -- {getattr(ENGINE, 'placement', '?')}; MIXED_CPU_GPU can also "
            "produce dtype mismatches across the boundary.",
        ]
        print("\n" + "=" * 62)
        print("MODEL SANITY TEST FAILED")
        print("=" * 62)
        for line in SANITY["diagnosis"]:
            print(" -", line)
        print("=" * 62)
        return SANITY

    SANITY["passed"] = True
    print("\nTEST 1: PASSED (output is real text, not a degenerate token run)")
    return SANITY


SANITY = run_sanity_gate(CFG)
MODEL_OCR_SANITY_CHECK = "PASSED" if SANITY.get("passed") else "FAILED"
print(f"\nMODEL_OCR_SANITY_CHECK = {MODEL_OCR_SANITY_CHECK}")

In [ ]:
# =========================================================================
# 17 — CUSTOMER AND DOCUMENT DISCOVERY + DUPLICATE SELECTION
# =========================================================================
def strip_accents(t: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", str(t)) if not unicodedata.combining(c))


def norm_name(t: str) -> str:
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9]+", " ", strip_accents(t).upper())).strip()


DOC_ALIASES = OrderedDict([
    (IDENTITY_DOC, ["JUSTIFICATIF IDENTITE", "JUSTIFICATIF D IDENTITE", "PIECE IDENTITE",
                    "JUSTIF IDENTITE", "IDENTITE"]),
    (DOMICILE_DOC, ["JUSTIFICATIF DOMICILE", "JUSTIFICATIF DE DOMICILE", "DOMICILE"]),
    (CONVENTION_DOC, ["CONVENTION COMPTE", "CONVENTION DE COMPTE", "OUVERTURE COMPTE"]),
    (FATCA_DOC, ["FATCA", "FORMULAIRE FATCA", "FATCA CRS"]),
    (SIGNATURE_DOC, ["CARTON SIGNATURE", "CARTON SIGNATUTE", "CARTON DE SIGNATURE",
                     "SPECIMEN SIGNATURE"]),
])
DOC_NORM = {d: sorted({norm_name(a) for a in [d] + al}, key=len, reverse=True)
            for d, al in DOC_ALIASES.items()}
DUP_RX = re.compile(r"\s*\((\d+)\)\s*$")


def classify_filename(name: str) -> Tuple[Optional[str], str]:
    p = Path(name)
    if p.suffix.lower() != ".pdf":
        return None, "not_pdf"
    stem = norm_name(DUP_RX.sub("", p.stem))
    for doc, aliases in DOC_NORM.items():
        if stem in aliases:
            return doc, "exact"
    for doc, aliases in DOC_NORM.items():
        for a in aliases:
            if len(a) >= 5 and a in stem:
                return doc, "contains"
    return None, "no_match"


def ensure_customer_root(cfg: Config = CFG) -> Path:
    root = Path(cfg.CUSTOMER_ROOT)
    if root.exists() and any(p.is_dir() for p in root.iterdir()):
        return root
    if cfg.ZIP_PATH and Path(cfg.ZIP_PATH).exists():
        root.mkdir(parents=True, exist_ok=True)
        log.info("extracting %s -> %s", cfg.ZIP_PATH, root)
        with zipfile.ZipFile(cfg.ZIP_PATH) as zf:
            for info in zf.infolist():
                if info.is_dir():
                    continue
                parts = [x for x in info.filename.replace("\\", "/").split("/") if x not in ("", ".")]
                if not parts or ".." in parts:
                    continue
                dest = root.joinpath(*parts)
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(zf.read(info))
        subs = [p for p in root.iterdir() if p.is_dir()]
        if len(subs) == 1 and any(c.is_dir() for c in subs[0].iterdir()):
            return subs[0]
        return root
    raise FileNotFoundError(f"neither CUSTOMER_ROOT {root} nor ZIP_PATH {cfg.ZIP_PATH} usable")


@dataclass
class CustomerDocs:
    customer_id: str
    selected: Dict[str, Optional[Path]]
    duplicates: Dict[str, List[str]]
    n_files: int

    @property
    def has_identity(self) -> bool:
        return self.selected.get(IDENTITY_DOC) is not None


def discover_customers(root: Path, cfg: Config = CFG) -> List[CustomerDocs]:
    """One representative PDF per logical document type; duplicates recorded, not processed."""
    out = []
    for d in sorted([p for p in Path(root).iterdir() if p.is_dir()], key=lambda p: p.name):
        pdfs = [p for p in d.rglob("*") if p.suffix.lower() == ".pdf"]
        by_doc: Dict[str, List[Path]] = defaultdict(list)
        for p in pdfs:
            doc, _ = classify_filename(p.name)
            if doc:
                by_doc[doc].append(p)
        selected, dups = {}, {}
        for doc in DOCUMENTS_REQUIRED:
            cands = by_doc.get(doc, [])
            if not cands:
                selected[doc], dups[doc] = None, []
                continue
            # deterministic: canonical filename first, then lowest copy index, then name
            def rank(p: Path):
                stem = p.stem
                m = DUP_RX.search(stem)
                return (0 if norm_name(stem) == norm_name(Path(doc).stem) else 1,
                        int(m.group(1)) if m else 0, p.name)
            ranked = sorted(cands, key=rank)
            selected[doc] = ranked[0]
            dups[doc] = [p.name for p in ranked[1:]]
        out.append(CustomerDocs(d.name, selected, dups, len(pdfs)))
    return out


def select_targets(customers: List[CustomerDocs], cfg: Config = CFG) -> List[CustomerDocs]:
    pool = [c for c in customers if c.has_identity]
    if cfg.PIN_CUSTOMER:
        pool = [c for c in pool if c.customer_id == cfg.PIN_CUSTOMER] or pool
    elif cfg.BENCHMARK_MODE:
        rng = random.Random(cfg.RANDOM_SEED)
        rng.shuffle(pool)
    return pool[:cfg.BENCHMARK_CUSTOMERS] if cfg.BENCHMARK_MODE else pool


def build_presence_report(customers: List[CustomerDocs]) -> pd.DataFrame:
    rows = []
    for c in customers:
        row = {"customer_id": c.customer_id}
        for doc, col in PRESENCE_COLUMNS.items():
            row[col] = "TRUE" if c.selected.get(doc) else "FALSE"
            row[col.replace("_exists", "") + "_selected_pdf"] = (
                c.selected[doc].name if c.selected.get(doc) else "")
            row[col.replace("_exists", "") + "_duplicates"] = ";".join(c.duplicates.get(doc, []))
        row["n_files_in_folder"] = c.n_files
        rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(DIRS["reports"] / "document_presence_report.csv", index=False, encoding="utf-8-sig")
    return df


print("17 ready: ensure_customer_root(), discover_customers(), select_targets()")

In [ ]:
# =========================================================================
# 18 / 19 — PAGE RENDERING (once), CACHE, AND CHEAP MRZ LOCALISATION
# =========================================================================
@dataclass
class RenderedPage:
    page_number: int
    image: Image.Image
    width: int
    height: int
    pdf_rect: Tuple[float, float, float, float]
    render_time_s: float
    text_layer: str = ""


class PdfCache:
    """Open once, render each ACTUAL page once, keep the handle for ROI clips."""

    def __init__(self, path: Path, cfg: Config = CFG):
        self.path, self.cfg, self.doc, self.pages = Path(path), cfg, None, []
        self.open_time_s = self.page_count_time_s = 0.0

    def __enter__(self):
        if pymupdf is None:
            raise RuntimeError("pymupdf is required")
        t0 = time.perf_counter()
        self.doc = pymupdf.open(str(self.path))
        self.open_time_s = round(time.perf_counter() - t0, 4)
        t0 = time.perf_counter()
        self.page_count = len(self.doc)                 # ACTUAL count, never assumed
        self.page_count_time_s = round(time.perf_counter() - t0, 4)
        for i in range(min(self.page_count, self.cfg.MAX_PAGES_PER_PDF)):
            t0 = time.perf_counter()
            page = self.doc[i]
            pix = page.get_pixmap(dpi=self.cfg.RENDER_DPI, colorspace=pymupdf.csRGB, alpha=False)
            img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples).copy()
            if max(img.size) > self.cfg.MAX_IMAGE_DIMENSION:
                s = self.cfg.MAX_IMAGE_DIMENSION / max(img.size)
                img = img.resize((int(img.width * s), int(img.height * s)), Image.LANCZOS)
            r = page.rect
            try:
                txt = page.get_text("text") or ""
            except Exception:
                txt = ""
            self.pages.append(RenderedPage(i + 1, img, img.width, img.height,
                                           (r.x0, r.y0, r.x1, r.y1),
                                           round(time.perf_counter() - t0, 3), txt))
        return self

    def __exit__(self, *exc):
        if self.doc is not None:
            self.doc.close()
            self.doc = None
        return False

    def clip(self, page_number: int, rect, dpi: int) -> Optional[Image.Image]:
        """Re-render a rectangle at high DPI through the OPEN handle (real resolution)."""
        if self.doc is None:
            return None
        try:
            page = self.doc[page_number - 1]
            r = pymupdf.Rect(*rect) & page.rect
            if r.is_empty:
                return None
            z = dpi / 72.0
            pix = page.get_pixmap(matrix=pymupdf.Matrix(z, z), clip=r,
                                  colorspace=pymupdf.csRGB, alpha=False)
            return Image.frombytes("RGB", (pix.width, pix.height), pix.samples).copy()
        except Exception:
            return None


MRZ_LINE_RX = re.compile(r"^[A-Z0-9<]{25,50}$")


def detect_mrz_band(img: Image.Image, cfg: Config = CFG) -> Optional[Dict[str, Any]]:
    """Cheap CV localisation of an MRZ band. No Qwen call is used to ask whether one exists.

    Discriminators against a photo, Arabic text, a printed field or security graphics:
    full width, high aspect ratio, dense fill, uniform glyph height, 2-3 text lines."""
    if cv2 is None or not cfg.USE_CV_PREPROCESSING:
        return None
    W0, H0 = img.size
    s = min(1.0, 1100 / W0)
    gray = np.asarray(img.convert("L").resize((int(W0 * s), int(H0 * s)), Image.BILINEAR))
    H, W = gray.shape
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (max(9, W // 60), 5))
    bh = cv2.morphologyEx(cv2.GaussianBlur(gray, (3, 3), 0), cv2.MORPH_BLACKHAT, k)
    g = np.absolute(cv2.Sobel(bh, cv2.CV_32F, 1, 0, ksize=-1))
    mn, mx = float(g.min()), float(g.max())
    g = ((g - mn) / (mx - mn + 1e-6) * 255).astype("uint8")
    g = cv2.morphologyEx(g, cv2.MORPH_CLOSE, k)
    th = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    th = cv2.erode(cv2.morphologyEx(th, cv2.MORPH_CLOSE,
                                    cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))),
                   None, iterations=2)
    th[:, :int(0.02 * W)] = 0
    th[:, int(0.98 * W):] = 0
    best = None
    for c in cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]:
        x, y, w, h = cv2.boundingRect(c)
        if h < 4 or w < 0.45 * W or w / float(h) < 4.0 or h > 0.30 * H:
            continue
        roi = th[y:y + h, x:x + w]
        fill = float((roi > 0).mean())
        if fill < 0.25:
            continue
        prof = (roi > 0).sum(axis=1).astype(np.float32)
        lines, run = 0, False
        for v in prof:
            hot = v > 0.35 * w
            if hot and not run:
                lines, run = lines + 1, True
            elif not hot:
                run = False
        score = min(w / float(h) / 20.0, 1.0) * 0.3 + (w / W) * 0.3 + fill * 0.2 + \
                (0.2 if lines in (2, 3) else 0.0)
        cand = {"bbox": [int(x / s), int(y / s), int((x + w) / s), int((y + h) / s)],
                "score": round(float(score), 3), "n_lines": int(lines),
                "fill": round(fill, 3), "width_frac": round(w / W, 3)}
        if best is None or cand["score"] > best["score"]:
            best = cand
    return best if (best and best["score"] >= 0.45) else None


print("18/19 ready: PdfCache, detect_mrz_band()")

In [ ]:
# =========================================================================
# 20 — PROMPTS FOR ALGERIAN KYC DOCUMENTS
# =========================================================================
# One page = one call = one full transcription. Structured fields are parsed from that text in
# Python (section 21), so there is NO separate Qwen call per field, per title or per name.
ALGERIAN_NOTE = """These are Algerian KYC documents: French administrative wording, often with Arabic alongside, frequently photocopied, stamped, skewed or low-contrast.
Common labels: Nom / اللقب, Prénom(s) / الاسم, Né(e) le, Date de naissance, Lieu de naissance, Commune, Daïra, Wilaya, Nationalité, Sexe, N° du document, Délivré le, Expire le, Autorité de délivrance, N° d'identification nationale (NIN), Nom et Prénom ou Raison Sociale, Numéro de compte, SPECIMEN DE SIGNATURE, FATCA.
Algerian names often have several components (BEN MOHAMED, ABD EL KADER, OULD KACI). Transcribe every component exactly as written, in the written order."""

PAGE_OCR_SYSTEM = f"""You transcribe scanned Algerian administrative and KYC documents. The image is the only source of truth.

{ALGERIAN_NOTE}

RULES
- Transcribe ALL text that is visibly readable: printed, handwritten, labels, headings, stamps.
- Do not guess, infer, reconstruct, complete, correct or reorder anything.
- Do not translate. Do not transliterate. Keep Arabic in Arabic, French in French, Latin names in Latin.
- Keep dates and numbers exactly as written. Do not reformat them.
- Preserve < and every MRZ character exactly if an MRZ is present.
- Mark unreadable text as [UNREADABLE]. Never invent its content.
- Output the transcription only: no preamble, no commentary, no summary, no layout description."""

PAGE_OCR_USER = "Transcribe everything written on this page, in reading order."

MRZ_SYSTEM = """You transcribe the machine readable zone (MRZ) of an identity document. The image is the only source of truth.
Preserve every character, every <, the character order and the line order. Insert no spaces.
Do not repair, translate or reorder. Check digits must NOT be used to decide an unclear character.
Mark an unreadable character with ?. Output the MRZ lines only, one per line, nothing else."""

MRZ_USER = "Transcribe the MRZ lines in this cropped image."

print("20 ready: page OCR prompt (Algerian context), MRZ prompt")
print(f"  system prompt: {len(PAGE_OCR_SYSTEM)} chars")

In [ ]:
# =========================================================================
# 21 — STRUCTURED EXTRACTION FROM THE TRANSCRIPTION  (pure Python, zero Qwen calls)
# =========================================================================
# This is the architectural decision that removes the inference explosion. The model reads each
# page once; every identity field, name and MRZ line is then recovered from THAT TEXT by label
# matching. Nothing here contacts the model, the database or another document.
FIELD_LABELS: Dict[str, List[str]] = {
    "surname_latin": ["NOM", "SURNAME", "NOM DE FAMILLE", "FAMILY NAME", "LAST NAME"],
    "given_names_latin": ["PRENOM", "PRENOMS", "GIVEN NAME", "GIVEN NAMES", "FIRST NAME"],
    "date_of_birth": ["DATE DE NAISSANCE", "DATE OF BIRTH", "NE LE", "NEE LE", "NE E LE"],
    "place_of_birth": ["LIEU DE NAISSANCE", "PLACE OF BIRTH", "COMMUNE DE NAISSANCE"],
    "nationality": ["NATIONALITE", "NATIONALITY"],
    "sex": ["SEXE", "SEX", "GENRE"],
    "document_number": ["N DU DOCUMENT", "NUMERO DU DOCUMENT", "DOCUMENT NO", "PASSPORT NO",
                        "N DE LA CARTE", "NUMERO", "N"],
    "issue_date": ["DELIVRE LE", "DATE DE DELIVRANCE", "DATE OF ISSUE"],
    "expiry_date": ["EXPIRE LE", "DATE D EXPIRATION", "VALABLE JUSQU AU", "DATE OF EXPIRY"],
    "issuing_authority": ["AUTORITE", "DELIVRE PAR", "ISSUING AUTHORITY", "DAIRA", "WILAYA"],
    "personal_number": ["N D IDENTIFICATION NATIONALE", "NIN", "NUMERO D IDENTIFICATION",
                        "PERSONAL NO"],
}
LABEL_INDEX = sorted(((lab, f) for f, labs in FIELD_LABELS.items() for lab in labs),
                     key=lambda x: -len(x[0]))
SIGNATURE_ANCHORS = ["NOM ET PRENOM OU RAISON SOCIALE", "NUMERO DE COMPTE"]
FATCA_ANCHORS = ["NOM ET PRENOM", "NOM", "PRENOM"]


def _label_norm(s: str) -> str:
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9 ]+", " ", strip_accents(s).upper())).strip()


def mrz_lines_from_text(raw: str) -> Optional[str]:
    """MRZ lines recovered from the page transcription. Returned EXACTLY as transcribed."""
    lines = [l.strip().replace(" ", "") for l in str(raw).splitlines() if l.strip()]
    hits = [l for l in lines if MRZ_LINE_RX.match(l) and l.count("<") >= 3]
    return "\n".join(hits[:3]) if len(hits) >= 2 else None


def parse_fields_from_text(raw: str, page: int, source: str = "page_ocr") -> Dict[str, Any]:
    """Label-driven field recovery. Values are copied verbatim from the transcription."""
    out: Dict[str, Any] = {}
    lines = [l for l in str(raw).splitlines() if l.strip()]
    norms = [_label_norm(l) for l in lines]
    for i, (line, nline) in enumerate(zip(lines, norms)):
        for label, fname in LABEL_INDEX:
            if fname in out or not nline.startswith(label):
                continue
            m = re.search(r"[:\-]\s*", line)
            value = (line[m.end():] if m else line[len(label):]).strip(" .:-\t")
            if not value and i + 1 < len(lines):
                nxt = lines[i + 1].strip()
                if nxt and not any(_label_norm(nxt).startswith(l2) for l2, _ in LABEL_INDEX):
                    value = nxt
            if value:
                out[fname] = {"value": value, "confidence": 0.75, "status": "READ",
                              "page": page, "region": source, "source_line": line}
            break
    mrz = mrz_lines_from_text(raw)
    if mrz:
        out["mrz"] = {"value": mrz, "confidence": 0.8, "status": "READ", "page": page,
                      "region": "page_ocr_mrz_lines", "source_line": None}
    return out


def anchored_value(raw: str, anchors: Sequence[str]) -> Optional[Dict[str, Any]]:
    """Value written after / below one of the given printed labels (FATCA, signature card)."""
    lines = [l for l in str(raw).splitlines() if l.strip()]
    norms = [_label_norm(l) for l in lines]
    for i, (line, nline) in enumerate(zip(lines, norms)):
        for a in anchors:
            na = _label_norm(a)
            if not nline.startswith(na):
                continue
            rest = line[len(a):].strip(" .:-\t") if len(line) > len(a) else ""
            if rest:
                return {"value": rest, "anchor": a, "position": "after", "line": line}
            if i + 1 < len(lines) and lines[i + 1].strip():
                return {"value": lines[i + 1].strip(), "anchor": a, "position": "below",
                        "line": lines[i + 1]}
    return None


def field_record(value, page, region, task, conf=0.75, status="READ", raw=None) -> Dict[str, Any]:
    return {"value": value, "confidence": conf if value else 0.0,
            "status": status if value else "NOT_FOUND", "page": page, "region": region,
            "ocr_task": task, "raw_model_output": (raw or "")[:600]}


def empty_field(status="NOT_FOUND") -> Dict[str, Any]:
    return {"value": None, "confidence": 0.0, "status": status, "page": None, "region": None,
            "ocr_task": None, "raw_model_output": ""}


print("21 ready: parse_fields_from_text(), mrz_lines_from_text(), anchored_value()  [0 Qwen calls]")

In [ ]:
# =========================================================================
# 22 — QWEN CALL WRAPPER  (counted, timed, limited, logged)
# =========================================================================
INFERENCE_LOG: List[Dict[str, Any]] = []
STAGE_TIMES: Dict[str, float] = defaultdict(float)
STAGE_CALLS: Dict[str, int] = defaultdict(int)
QWEN_CALLS = {"total": 0}
RUN_DEADLINE = {"t": None}


class stage:
    """Cheap hierarchical timer."""

    def __init__(self, name: str, sink: Optional[Dict[str, float]] = None):
        self.name, self.sink = name, sink

    def __enter__(self):
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        dt = time.perf_counter() - self.t0
        STAGE_TIMES[self.name] += dt
        STAGE_CALLS[self.name] += 1
        if self.sink is not None:
            self.sink[self.name] = self.sink.get(self.name, 0.0) + dt
        return False


def qwen_call(image: Image.Image, system: str, user: str, max_new_tokens: int, *,
              customer_id: str, document: str, page: Optional[int], task: str,
              roi: str = "full_page", retry: int = 0, cfg: Config = CFG) -> Dict[str, Any]:
    """The ONLY place the model is invoked. Every call is counted, timed and logged."""
    if cfg.DRY_RUN:
        raise RuntimeError("DRY_RUN is on: inference must not execute. Set CFG.DRY_RUN=False.")
    if MODEL_OCR_SANITY_CHECK != "PASSED":
        raise RuntimeError("MODEL_OCR_SANITY_CHECK failed: refusing to run the pipeline.")
    if RUN_DEADLINE["t"] and time.perf_counter() > RUN_DEADLINE["t"]:
        raise TimeoutError(f"MAX_RUNTIME_SECONDS exceeded at customer={customer_id} "
                           f"document={document} page={page} task={task} "
                           f"call={QWEN_CALLS['total']}")
    QWEN_CALLS["total"] += 1
    n = QWEN_CALLS["total"]
    t0 = time.perf_counter()
    r = raw_generate(image, system, user, max_new_tokens, cfg)
    elapsed = time.perf_counter() - t0
    STAGE_TIMES["qwen_generation"] += r.get("generation_time_s", 0.0)
    STAGE_TIMES["processor"] += r.get("processor_time_s", 0.0)
    STAGE_CALLS["qwen_generation"] += 1

    degen = _degenerate(r.get("text", ""), r.get("ids") or None) if r["status"] == "OK" else None
    mem = gpu_snapshot(f"qwen_{n}") if n % 10 == 1 else {}
    INFERENCE_LOG.append({
        "timestamp": utcnow(), "qwen_call": n, "customer": customer_id, "document": document,
        "page": page, "task": task, "roi_type": roi, "image_width": image.width,
        "image_height": image.height, "input_tokens": r.get("input_tokens"),
        "visual_tokens": r.get("visual_tokens"), "output_tokens": r.get("output_tokens"),
        "max_new_tokens": max_new_tokens, "retry_number": retry,
        "start_time": round(t0, 3), "end_time": round(time.perf_counter(), 3),
        "generation_time_s": r.get("generation_time_s"), "elapsed_s": round(elapsed, 3),
        "tokens_per_s": r.get("tokens_per_s"), "truncated": r.get("truncated"),
        "degenerate": degen, "status": "DEGENERATE" if degen else r["status"],
        "error": r.get("error"), "model": getattr(ENGINE, "name", "?"),
        "attention": getattr(ENGINE, "attn", "?"),
        "gpu_peak_gib": mem.get("torch_peak_allocated_gib")})

    print(f"[QWEN {n}] customer={customer_id} document={document} page={page} task={task} "
          f"ROI={roi} input={image.width}x{image.height} out={r.get('output_tokens')}tok "
          f"mnt={max_new_tokens} elapsed={elapsed:.1f}s "
          f"tok/s={r.get('tokens_per_s')}", flush=True)
    if elapsed > cfg.WARN_QWEN_CALL_SECONDS:
        print(f"  WARNING: SLOW QWEN CALL ({elapsed:.1f}s) visual_tokens={r.get('visual_tokens')} "
              f"placement={getattr(ENGINE, 'placement', '?')}", flush=True)
    if degen:
        print(f"  WARNING: DEGENERATE OUTPUT -- {degen}", flush=True)
    r["degenerate"] = degen
    r["qwen_call"] = n
    return r


print("22 ready: qwen_call()")

In [ ]:
# =========================================================================
# 23 — TASK PLANNER AND DRY RUN
# =========================================================================
# Planned calls = actual pages + (0-1 MRZ recovery per identity document).
@dataclass
class PlannedTask:
    customer_id: str
    document: str
    pdf: str
    page: Optional[int]
    task: str
    roi_type: str
    reason: str
    estimated_max_new_tokens: int


def plan_customer(cust: CustomerDocs, cfg: Config = CFG
                  ) -> Tuple[List[PlannedTask], List[Dict[str, Any]]]:
    """Opens each selected PDF once to read the REAL page count. No inference occurs."""
    tasks, rows = [], []
    for doc in DOCUMENTS_REQUIRED:
        path = cust.selected.get(doc)
        if not path:
            rows.append({"customer_id": cust.customer_id, "document": doc, "pdf": None,
                         "page_count": 0, "planned_calls": 0, "duplicates": "",
                         "note": "missing"})
            continue
        try:
            with PdfCache(path, cfg) as pc:
                n = pc.page_count
                for p in pc.pages:
                    tasks.append(PlannedTask(cust.customer_id, doc, path.name, p.page_number,
                                             "page_ocr", "full_page",
                                             "one transcription per actual page",
                                             cfg.MAX_NEW_TOKENS))
                mrz_pages = []
                if doc == IDENTITY_DOC:
                    for p in pc.pages:
                        if detect_mrz_band(p.image, cfg):
                            mrz_pages.append(p.page_number)
                    if mrz_pages:
                        tasks.append(PlannedTask(
                            cust.customer_id, doc, path.name, mrz_pages[0], "mrz_ocr", "mrz_band",
                            f"MRZ band detected on page(s) {mrz_pages}; used only if the page "
                            f"transcription does not already contain readable MRZ lines",
                            cfg.MAX_NEW_TOKENS_MRZ))
                planned = len([t for t in tasks if t.document == doc
                               and t.customer_id == cust.customer_id])
                rows.append({"customer_id": cust.customer_id, "document": doc, "pdf": path.name,
                             "page_count": n, "planned_calls": planned,
                             "duplicates": ";".join(cust.duplicates.get(doc, [])),
                             "mrz_candidate_pages": ";".join(map(str, mrz_pages)), "note": ""})
        except Exception as exc:
            rows.append({"customer_id": cust.customer_id, "document": doc,
                         "pdf": path.name if path else None, "page_count": -1,
                         "planned_calls": 0, "duplicates": "",
                         "note": f"open failed: {type(exc).__name__}: {exc}"})
    return tasks, rows


class TaskExplosion(RuntimeError):
    pass


def validate_plan(tasks_df: pd.DataFrame, cfg: Config = CFG) -> None:
    if tasks_df.empty:
        print("plan check: nothing planned")
        return
    problems = []
    if len(tasks_df) > cfg.MAX_TOTAL_QWEN_CALLS:
        problems.append(f"total {len(tasks_df)} > MAX_TOTAL_QWEN_CALLS {cfg.MAX_TOTAL_QWEN_CALLS}")
    for cid, n in tasks_df.groupby("customer_id").size().items():
        if n > cfg.MAX_QWEN_CALLS_PER_CUSTOMER:
            problems.append(f"customer {cid}: {n} > MAX_QWEN_CALLS_PER_CUSTOMER "
                            f"{cfg.MAX_QWEN_CALLS_PER_CUSTOMER}")
    for (cid, doc), n in tasks_df.groupby(["customer_id", "document"]).size().items():
        if n > cfg.MAX_QWEN_CALLS_PER_PDF:
            problems.append(f"{cid}/{doc}: {n} > MAX_QWEN_CALLS_PER_PDF "
                            f"{cfg.MAX_QWEN_CALLS_PER_PDF}")
    if problems:
        for p in problems[:10]:
            print("  !!", p)
        raise TaskExplosion(f"{len(problems)} limit violation(s); nothing was sent to the GPU. "
                            "Review execution_plan.csv or raise the ceiling deliberately.")
    print(f"plan check: OK ({len(tasks_df)} planned calls)")


def print_plan(tasks_df: pd.DataFrame, docs_df: pd.DataFrame, cfg: Config = CFG) -> None:
    print("=" * 50)
    print("INFERENCE PLAN")
    print("=" * 50)
    for cid, grp in docs_df.groupby("customer_id"):
        print(f"\ncustomer: {cid}")
        for _, r in grp.iterrows():
            dup = f"  (+{len(str(r['duplicates']).split(';'))} duplicate ignored)" \
                if r["duplicates"] else ""
            note = f"  [{r['note']}]" if r["note"] else ""
            print(f"  {r['document']:<28} pages={int(r['page_count']):<3} "
                  f"calls={int(r['planned_calls'])}{dup}{note}")
    n_calls = len(tasks_df)
    n_pages = int(docs_df["page_count"].clip(lower=0).sum())
    print(f"\nPDFs:                {int((docs_df['page_count'] > 0).sum())}")
    print(f"pages:               {n_pages}")
    print(f"planned Qwen calls:  {n_calls}")
    print(f"maximum allowed:     {cfg.MAX_QWEN_CALLS_PER_CUSTOMER} per customer / "
          f"{cfg.MAX_QWEN_CALLS_PER_PDF} per PDF")
    rate = MEASURED_RATE["tok_s"]
    if rate:
        est = tasks_df["estimated_max_new_tokens"].sum() / rate
        print(f"estimated runtime:   <= {est/60:.1f} min at the measured {rate} tok/s "
              f"(calls stop early at EOS)")
    else:
        print("estimated runtime:   run the sanity gate to measure the decode rate")
    print("=" * 50)


MEASURED_RATE = {"tok_s": None}
if SANITY.get("tests", {}).get("test1", {}).get("tokens_per_s"):
    MEASURED_RATE["tok_s"] = SANITY["tests"]["test1"]["tokens_per_s"]

print("23 ready: plan_customer(), validate_plan(), print_plan()")

In [ ]:
# =========================================================================
# 24-28 — DOCUMENT PROCESSING  (one call per page; fields parsed from the text)
# =========================================================================
EXPECTED_TITLES = {
    IDENTITY_DOC: ["CARTE NATIONALE D IDENTITE", "CARTE D IDENTITE", "PASSEPORT", "PASSPORT",
                   "PERMIS DE CONDUIRE", "REPUBLIQUE ALGERIENNE"],
    DOMICILE_DOC: ["FACTURE", "QUITTANCE", "ATTESTATION DE RESIDENCE", "CERTIFICAT DE RESIDENCE",
                   "SONELGAZ", "ALGERIE TELECOM", "SEAAL"],
    CONVENTION_DOC: ["CONVENTION DE COMPTE", "CONVENTION COMPTE", "OUVERTURE DE COMPTE"],
    FATCA_DOC: ["FATCA IDENTIFICATION FORMS",
                "FORMULAIRE D IDENTIFICATION", "AU REGARD DE LA LOI FATCA"],
    SIGNATURE_DOC: ["SPECIMEN DE SIGNATURE"],      # correct spelling; SPICIMEN is NOT expected
}


def verify_title(all_text: str, doc: str) -> Dict[str, Any]:
    hay = _label_norm(all_text)
    for phrase in EXPECTED_TITLES.get(doc, []):
        if _label_norm(phrase) in hay:
            return {"status": "MATCH", "matched": phrase}
    if not hay.strip():
        return {"status": "UNCERTAIN", "matched": None, "reason": "no readable text"}
    return {"status": "MISMATCH", "matched": None,
            "reason": "none of the expected titles appear in the transcription"}


def process_document(cust: CustomerDocs, doc: str, path: Path, cfg: Config = CFG
                     ) -> Dict[str, Any]:
    """Transcribe every ACTUAL page once, then parse. One extra call only to recover an MRZ."""
    t_doc = time.perf_counter()
    timings: Dict[str, float] = {}
    pages_out, errors, raw_all = [], [], []
    fields: Dict[str, Any] = {}
    mrz_report = {"mrz_detected": False, "mrz_page": None, "mrz_source": None,
                  "mrz_region": None, "mrz_calls": 0}

    with PdfCache(path, cfg) as pc:
        timings["pdf_open"] = pc.open_time_s
        timings["page_count"] = pc.page_count_time_s
        for page in pc.pages:
            timings["render"] = timings.get("render", 0.0) + page.render_time_s
            with stage("mrz_detect", timings):
                band = detect_mrz_band(page.image, cfg) if doc == IDENTITY_DOC else None
            r = qwen_call(page.image, PAGE_OCR_SYSTEM, PAGE_OCR_USER, cfg.MAX_NEW_TOKENS,
                          customer_id=cust.customer_id, document=doc, page=page.page_number,
                          task="page_ocr", roi="full_page", cfg=cfg)
            raw = r.get("text", "")
            raw_all.append(raw)
            with stage("parse", timings):
                parsed = parse_fields_from_text(raw, page.page_number)
            for k, v in parsed.items():
                if k not in fields or fields[k].get("value") is None:
                    fields[k] = v
            if band:
                mrz_report.update({"mrz_detected": True, "mrz_page": page.page_number,
                                   "mrz_region": band["bbox"], "mrz_detector_score": band["score"]})
            txt_path = DIRS["raw"] / f"{cust.customer_id}__{Path(doc).stem}__p{page.page_number:03d}.txt"
            with stage("io", timings):
                txt_path.write_text(raw, encoding="utf-8")     # raw, unmodified
            pages_out.append({
                "customer_id": cust.customer_id, "document": doc, "pdf": path.name,
                "page_number": page.page_number, "page_count": pc.page_count,
                "image_width": page.width, "image_height": page.height,
                "render_time_s": page.render_time_s, "qwen_call": r.get("qwen_call"),
                "generation_time_s": r.get("generation_time_s"),
                "output_tokens": r.get("output_tokens"), "visual_tokens": r.get("visual_tokens"),
                "degenerate": r.get("degenerate"), "status": r.get("status"),
                "has_text_layer": bool(page.text_layer.strip()),
                "raw_text_path": str(txt_path), "raw_text": raw})
            if r.get("degenerate"):
                errors.append({"timestamp": utcnow(), "customer_id": cust.customer_id,
                               "document": doc, "page": page.page_number, "stage": "page_ocr",
                               "error": f"degenerate output: {r['degenerate']}"})

        # ---- MRZ recovery: ONE extra call, only if the page text lacks readable MRZ lines ----
        if doc == IDENTITY_DOC and mrz_report["mrz_detected"] and \
                fields.get("mrz", {}).get("value") is None and cfg.MAX_TARGETED_RETRIES > 0:
            page = next(p for p in pc.pages if p.page_number == mrz_report["mrz_page"])
            x0, y0, x1, y1 = mrz_report["mrz_region"]
            fx0, fy0 = x0 / page.width, y0 / page.height
            fx1, fy1 = x1 / page.width, y1 / page.height
            px0, py0, px1, py1 = page.pdf_rect
            pw, ph = px1 - px0, py1 - py0
            rect = (px0 + fx0 * pw, py0 + fy0 * ph - 0.01 * ph,
                    px0 + fx1 * pw, py0 + fy1 * ph + 0.01 * ph)
            with stage("mrz_crop", timings):
                crop = pc.clip(mrz_report["mrz_page"], rect, cfg.REGION_RENDER_DPI)
                if crop is None:
                    crop = page.image.crop((x0, y0, x1, y1))
                    mrz_report["mrz_source"] = "pixel_crop"
                else:
                    mrz_report["mrz_source"] = "pdf_clip_rerender"
                if crop.width < 1400:
                    f = min(cfg.MRZ_UPSCALE, 1600 / max(1, crop.width))
                    crop = crop.resize((int(crop.width * f), int(crop.height * f)), Image.LANCZOS)
            rm = qwen_call(crop, MRZ_SYSTEM, MRZ_USER, cfg.MAX_NEW_TOKENS_MRZ,
                           customer_id=cust.customer_id, document=doc,
                           page=mrz_report["mrz_page"], task="mrz_ocr", roi="mrz_band",
                           retry=1, cfg=cfg)
            mrz_report["mrz_calls"] = 1
            mrz_text = mrz_lines_from_text(rm.get("text", "")) or (
                rm.get("text") if MRZ_LINE_RX.match((rm.get("text") or "").strip().split("\n")[0]
                                                    or "x") else None)
            if mrz_text:
                fields["mrz"] = field_record(mrz_text, mrz_report["mrz_page"], "mrz_band",
                                             "mrz_ocr", conf=0.8, raw=rm.get("text"))
            mrz_report["mrz_raw_output"] = (rm.get("text") or "")[:600]

    all_text = "\n".join(raw_all)
    title = verify_title(all_text, doc)

    # ---- document-specific name recovery, all from the same transcription ----
    names: Dict[str, Any] = {}
    if doc in (DOMICILE_DOC, CONVENTION_DOC):
        for f in NAME_FIELDS:
            names[f] = (field_record(fields[f]["value"], fields[f]["page"], "page_ocr",
                                     "page_ocr", raw=all_text[:600])
                        if f in fields else empty_field())
    elif doc == FATCA_DOC:
        a = anchored_value(all_text, ["NOM ET PRENOM"])
        nom = anchored_value(all_text, ["NOM"])
        pre = anchored_value(all_text, ["PRENOM", "PRENOMS"])
        raw_nom = (a or nom or {}).get("value")
        raw_prenom = (pre or {}).get("value")
        names = {
            "raw_nom": field_record(raw_nom, None, (a or nom or {}).get("anchor"), "page_ocr"),
            "raw_prenom": field_record(raw_prenom, None, (pre or {}).get("anchor"), "page_ocr"),
            "surname_latin": field_record(raw_nom, None, "fatca_anchor", "page_ocr"),
            "given_names_latin": field_record(raw_prenom, None, "fatca_anchor", "page_ocr"),
        }
        combined = " ".join([x for x in (raw_nom, raw_prenom) if x]) or None
        names["combined_name"] = field_record(combined, None, "fatca_anchor", "page_ocr")
    elif doc == SIGNATURE_DOC:
        a = anchored_value(all_text, SIGNATURE_ANCHORS)
        names = {"surname_latin": empty_field(), "given_names_latin": empty_field(),
                 "combined_name": field_record((a or {}).get("value"), None,
                                               (a or {}).get("anchor"), "page_ocr")}

    return {"document": doc, "pdf": str(path), "page_count": len(pages_out),
            "document_verification_status": title["status"], "title_check": title,
            "fields": fields, "names": names, "mrz": mrz_report, "pages": pages_out,
            "timings_seconds": {k: round(v, 3) for k, v in timings.items()},
            "errors": errors, "processing_time_s": round(time.perf_counter() - t_doc, 2)}


print("24-28 ready: process_document()")

In [ ]:
# =========================================================================
# 27 / 28 — CROSS-DOCUMENT AND DATABASE VERIFICATION  (CPU only, after extraction)
# =========================================================================
# Comparison NEVER writes into an extracted value. Unreadable is NOT a mismatch.
MATCH, MISMATCH, UNCERTAIN, NOT_AVAILABLE = "MATCH", "MISMATCH", "UNCERTAIN", "NOT_AVAILABLE"


def norm_cmp(v: Optional[str]) -> Optional[str]:
    """Comparison key only. Never stored in place of a raw value."""
    if not v:
        return None
    s = re.sub(r"[^A-Z ]+", " ", strip_accents(str(v)).upper())
    return re.sub(r"\s+", " ", s).strip() or None


def parse_date_any(v: Optional[str]):
    if not v:
        return None
    s = strip_accents(str(v)).strip()
    for rx, order in [(r"^(\d{2})[./\- ](\d{2})[./\- ](\d{4})$", "dmy"),
                      (r"^(\d{4})[./\- ](\d{2})[./\- ](\d{2})$", "ymd"),
                      (r"^(\d{2})[./\- ](\d{2})[./\- ](\d{2})$", "dmyy")]:
        m = re.match(rx, s)
        if not m:
            continue
        a, b, c = m.groups()
        try:
            if order == "dmy":
                return date(int(c), int(b), int(a))
            if order == "ymd":
                return date(int(a), int(b), int(c))
            y = int(c)
            return date(2000 + y if y < 30 else 1900 + y, int(b), int(a))
        except Exception:
            return None
    return None


def compare_text(doc_v, ref_v, cfg: Config = CFG) -> Dict[str, Any]:
    a, b = norm_cmp(doc_v), norm_cmp(ref_v)
    out = {"document_value": doc_v, "reference_value": ref_v, "normalized_document": a,
           "normalized_reference": b, "method": "normalized_exact", "score": None}
    if a is None or b is None:
        return {**out, "status": NOT_AVAILABLE,
                "reason": "missing on one side -- unreadable is not a mismatch"}
    if a == b:
        return {**out, "status": MATCH, "score": 1.0}
    score = round(difflib.SequenceMatcher(None, a, b).ratio(), 3)
    out.update({"method": "difflib_ratio", "score": score, "threshold": cfg.NAME_FUZZY_THRESHOLD})
    if score >= cfg.NAME_FUZZY_THRESHOLD:
        return {**out, "status": UNCERTAIN, "reason": "near match, not identical"}
    return {**out, "status": MISMATCH}


def compare_name_any_order(surname, given, db_name, cfg: Config = CFG) -> Dict[str, Any]:
    """`Nom abrégé tiers` may be LAST FIRST or FIRST LAST: compare as token multisets."""
    full = " ".join([x for x in (surname, given) if x]) or None
    out = {"document_value": full, "reference_value": db_name,
           "method": "token_multiset_any_order", "score": None, "observed_order": None}
    da, db = norm_cmp(full), norm_cmp(db_name)
    if not da or not db:
        return {**out, "status": NOT_AVAILABLE, "reason": "missing on one side"}
    ta, tb = sorted(da.split()), sorted(db.split())
    if ta == tb:
        order = None
        if surname:
            order = "LAST_FIRST" if db.startswith(norm_cmp(surname) or "") else "FIRST_LAST"
        return {**out, "status": MATCH, "score": 1.0, "observed_order": order}
    score = round(difflib.SequenceMatcher(None, " ".join(ta), " ".join(tb)).ratio(), 3)
    out["score"] = score
    if score >= cfg.NAME_FUZZY_THRESHOLD:
        return {**out, "status": UNCERTAIN, "reason": "near match, tokens differ"}
    return {**out, "status": MISMATCH}


def compare_dates(doc_v, ref_v) -> Dict[str, Any]:
    d1, d2 = parse_date_any(doc_v), parse_date_any(ref_v)
    out = {"document_value": doc_v, "reference_value": ref_v, "method": "parsed_date_equality",
           "score": None}
    if not doc_v or not ref_v:
        return {**out, "status": NOT_AVAILABLE}
    if d1 is None or d2 is None:
        return {**out, "status": UNCERTAIN, "reason": "unparsable on one side; raw preserved"}
    return {**out, "status": MATCH if d1 == d2 else MISMATCH, "score": 1.0 if d1 == d2 else 0.0}


def load_database(cfg: Config = CFG) -> Optional[pd.DataFrame]:
    """ONLY the four authorised columns are read; the rest of the file is discarded here."""
    p = cfg.CUSTOMER_DATABASE_PATH
    if not p or not Path(p).exists():
        log.warning("customer database not found: database verification -> NOT_AVAILABLE")
        return None
    raw = (pd.read_csv(p, dtype=str, sep=None, engine="python")
           if str(p).lower().endswith(".csv") else pd.read_excel(p, dtype=str)).fillna("")

    def find(col):
        key = lambda s: re.sub(r"[^a-z0-9]", "", strip_accents(str(s)).lower())
        for c in raw.columns:
            if key(c) == key(col):
                return c
        for c in raw.columns:
            if key(col) and key(col) in key(c):
                return c
        return None

    cols = {k: find(v) for k, v in [("customer_id", cfg.DB_COL_ID), ("name", cfg.DB_COL_NAME),
                                    ("date_of_birth", cfg.DB_COL_DOB),
                                    ("expiry_date", cfg.DB_COL_EXPIRY)]}
    if any(v is None for v in cols.values()):
        log.error("database columns not found: %s (available: %s)", cols, list(raw.columns))
        return None
    log.info("database columns in use (ONLY these): %s", cols)
    df = raw[[cols[k] for k in ("customer_id", "name", "date_of_birth", "expiry_date")]].copy()
    df.columns = ["customer_id", "name", "date_of_birth", "expiry_date"]
    df["customer_id"] = df["customer_id"].astype(str).str.strip()
    return df.set_index("customer_id", drop=False)


DB = load_database(CFG)
print("28 ready:", "database loaded" if DB is not None else "no database (verification NA)")

In [ ]:
# =========================================================================
# 32 — CUSTOMER PROCESSING AND AGGREGATION
# =========================================================================
def process_customer(cust: CustomerDocs, cfg: Config = CFG) -> Dict[str, Any]:
    t0 = time.perf_counter()
    calls_before = QWEN_CALLS["total"]
    documents: Dict[str, Optional[Dict[str, Any]]] = {}
    errors: List[Dict[str, Any]] = []
    for doc in DOCUMENTS_REQUIRED:
        path = cust.selected.get(doc)
        if not path:
            documents[doc] = None
            continue
        try:
            documents[doc] = process_document(cust, doc, path, cfg)
            errors.extend(documents[doc].get("errors", []))
        except TimeoutError:
            raise
        except Exception as exc:
            documents[doc] = None
            errors.append({"timestamp": utcnow(), "customer_id": cust.customer_id,
                           "document": doc, "page": None, "stage": "process_document",
                           "error": f"{type(exc).__name__}: {exc}",
                           "traceback": traceback.format_exc()})
            log.error("%s failed for %s: %s", doc, cust.customer_id, exc)

    identity = documents.get(IDENTITY_DOC) or {}
    id_fields = {f: (identity.get("fields", {}).get(f) or empty_field())
                 for f in IDENTITY_FIELDS}

    # ---- cross-document: compare only; never fill ----
    with stage("cross_document"):
        matrix, comparisons = [], []
        ref_sur = id_fields["surname_latin"].get("value")
        ref_giv = id_fields["given_names_latin"].get("value")
        for doc in DOCUMENTS_REQUIRED:
            res = documents.get(doc)
            if doc == IDENTITY_DOC:
                matrix.append({"document": doc, "surname_latin": "reference",
                               "given_names_latin": "reference", "overall": "reference",
                               "document_verification_status":
                                   (res or {}).get("document_verification_status", "MISSING")})
                continue
            if res is None:
                matrix.append({"document": doc, "surname_latin": NOT_AVAILABLE,
                               "given_names_latin": NOT_AVAILABLE, "overall": NOT_AVAILABLE,
                               "document_verification_status": "MISSING"})
                continue
            names = res.get("names", {})
            st = {}
            for f, ref in (("surname_latin", ref_sur), ("given_names_latin", ref_giv)):
                c = compare_text((names.get(f) or {}).get("value"), ref, cfg)
                c.update({"source_document": doc, "field": f, "reference": IDENTITY_DOC})
                comparisons.append(c)
                st[f] = c["status"]
            if (names.get("combined_name") or {}).get("value"):
                c = compare_name_any_order(ref_sur, ref_giv,
                                           names["combined_name"]["value"], cfg)
                c.update({"source_document": doc, "field": "combined_name",
                          "reference": IDENTITY_DOC})
                comparisons.append(c)
                if all(v == NOT_AVAILABLE for v in st.values()):
                    st = {"surname_latin": c["status"], "given_names_latin": c["status"]}
            overall = (MISMATCH if MISMATCH in st.values() else
                       UNCERTAIN if UNCERTAIN in st.values() else
                       NOT_AVAILABLE if all(v == NOT_AVAILABLE for v in st.values()) else MATCH)
            matrix.append({"document": doc, **st, "overall": overall,
                           "document_verification_status":
                               res.get("document_verification_status")})

    # ---- database: verification only, four columns ----
    with stage("database_match"):
        row = None
        if DB is not None and str(cust.customer_id) in DB.index:
            r0 = DB.loc[str(cust.customer_id)]
            row = None if isinstance(r0, pd.DataFrame) else {k: str(v).strip()
                                                             for k, v in r0.to_dict().items()}
        if row is None:
            na = {"status": NOT_AVAILABLE, "document_value": None, "reference_value": None,
                  "method": "none"}
            db_check = {"db_id": None, "comparison": {k: na for k in
                        ("customer_id", "name", "date_of_birth", "expiry_date")},
                        "overall": NOT_AVAILABLE}
        else:
            cid_cmp = {"document_value": cust.customer_id, "reference_value": row["customer_id"],
                       "method": "exact_string",
                       "status": MATCH if str(cust.customer_id).strip() ==
                                 str(row["customer_id"]).strip() else MISMATCH}
            cmp_ = {"customer_id": cid_cmp,
                    "name": compare_name_any_order(ref_sur, ref_giv, row["name"], cfg),
                    "date_of_birth": compare_dates(id_fields["date_of_birth"].get("value"),
                                                   row["date_of_birth"]),
                    "expiry_date": compare_dates(id_fields["expiry_date"].get("value"),
                                                 row["expiry_date"])}
            sts = [v["status"] for v in cmp_.values()]
            db_check = {"db_id": row["customer_id"], "comparison": cmp_,
                        "overall": (MISMATCH if MISMATCH in sts else
                                    UNCERTAIN if UNCERTAIN in sts else
                                    MATCH if MATCH in sts else NOT_AVAILABLE)}

    reasons = []
    missing_docs = [d for d in DOCUMENTS_REQUIRED if not cust.selected.get(d)]
    unreadable = [f for f in ("surname_latin", "given_names_latin", "date_of_birth")
                  if id_fields[f].get("value") is None]
    wrong_type = [m["document"] for m in matrix
                  if m.get("document_verification_status") == "MISMATCH"]
    name_mm = [m["document"] for m in matrix if m.get("overall") == MISMATCH]
    if missing_docs:
        reasons.append("missing_documents:" + ",".join(missing_docs))
    if unreadable:
        reasons.append("unreadable_identity_fields:" + ",".join(unreadable))
    if wrong_type:
        reasons.append("wrong_document_type:" + ",".join(wrong_type))
    if name_mm:
        reasons.append("name_mismatch:" + ",".join(name_mm))
    if db_check["overall"] == MISMATCH:
        reasons.append("database_mismatch")
    status = ("MISMATCH" if (name_mm or wrong_type or db_check["overall"] == MISMATCH) else
              "INCOMPLETE" if missing_docs else
              "UNCERTAIN" if (unreadable or db_check["overall"] != MATCH or reasons) else "PASS")

    return {"customer_id": cust.customer_id, "run_id": RUN_ID, "extracted_at_utc": utcnow(),
            "model": getattr(ENGINE, "name", "?"), "attention": getattr(ENGINE, "attn", "?"),
            "document_presence": {PRESENCE_COLUMNS[d]: bool(cust.selected.get(d))
                                  for d in DOCUMENTS_REQUIRED},
            "duplicates": cust.duplicates, "documents": documents,
            "identity_fields": id_fields, "mrz": identity.get("mrz", {}),
            "cross_document_verification": matrix, "cross_document_comparisons": comparisons,
            "database_verification": db_check,
            "overall_kyc_status": status, "reasons": reasons,
            "needs_manual_review": status != "PASS",
            "qwen_calls": QWEN_CALLS["total"] - calls_before,
            "page_count_total": sum((d or {}).get("page_count", 0) for d in documents.values()),
            "processing_time_s": round(time.perf_counter() - t0, 2), "errors": errors}


print("32 ready: process_customer()")

In [ ]:
# =========================================================================
# 33 / 34 — OUTPUTS AND PERFORMANCE REPORTS
# =========================================================================
RESULTS: List[Dict[str, Any]] = []
ALL_ERRORS: List[Dict[str, Any]] = []


def write_outputs(cfg: Config = CFG) -> Dict[str, Path]:
    rep, paths = DIRS["reports"], {}
    with (rep / "identity_extraction_results.jsonl").open("w", encoding="utf-8") as fh:
        for r in RESULTS:
            fh.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")
    paths["jsonl"] = rep / "identity_extraction_results.jsonl"

    rows = []
    for r in RESULTS:
        row = OrderedDict(customer_id=r["customer_id"],
                          overall_kyc_status=r["overall_kyc_status"],
                          needs_manual_review=r["needs_manual_review"],
                          reasons=";".join(r["reasons"]), qwen_calls=r["qwen_calls"],
                          pages=r["page_count_total"],
                          processing_time_s=r["processing_time_s"])
        row.update(r["document_presence"])
        for f in IDENTITY_FIELDS:
            n = r["identity_fields"].get(f) or {}
            row[f] = n.get("value")
            row[f + "_confidence"] = n.get("confidence")
            row[f + "_status"] = n.get("status")
            row[f + "_page"] = n.get("page")
        m = r.get("mrz") or {}
        row.update({"mrz_detected": m.get("mrz_detected"), "mrz_page": m.get("mrz_page"),
                    "mrz_source": m.get("mrz_source"), "mrz_extra_calls": m.get("mrz_calls")})
        for x in r["cross_document_verification"]:
            key = re.sub(r"[^a-z]+", "_", x["document"].lower()).strip("_")
            row[f"identity_vs_{key}"] = x.get("overall")
        for k, v in (r["database_verification"]["comparison"]).items():
            row[f"db_{k}_match"] = v["status"]
        rows.append(row)
    flat = pd.DataFrame(rows)
    flat.to_csv(rep / "identity_extraction_results.csv", index=False, encoding="utf-8-sig")
    paths["identity_csv"] = rep / "identity_extraction_results.csv"

    pd.DataFrame([{"customer_id": r["customer_id"], **m} for r in RESULTS
                  for m in r["cross_document_verification"]]).to_csv(
        rep / "cross_document_verification.csv", index=False, encoding="utf-8-sig")
    paths["cross_document"] = rep / "cross_document_verification.csv"

    dbrows = []
    for r in RESULTS:
        d = r["database_verification"]
        c = d["comparison"]
        dbrows.append({"customer_id": r["customer_id"], "db_id": d.get("db_id"),
                       "id_match_status": c["customer_id"]["status"],
                       "ocr_full_name": c["name"].get("document_value"),
                       "db_full_name": c["name"].get("reference_value"),
                       "name_match_status": c["name"]["status"],
                       "name_score": c["name"].get("score"),
                       "name_observed_order": c["name"].get("observed_order"),
                       "ocr_date_of_birth": c["date_of_birth"].get("document_value"),
                       "db_date_of_birth": c["date_of_birth"].get("reference_value"),
                       "dob_match_status": c["date_of_birth"]["status"],
                       "ocr_expiry_date": c["expiry_date"].get("document_value"),
                       "db_expiry_date": c["expiry_date"].get("reference_value"),
                       "expiry_match_status": c["expiry_date"]["status"],
                       "overall": d["overall"]})
    pd.DataFrame(dbrows).to_csv(rep / "database_verification.csv", index=False,
                                encoding="utf-8-sig")
    paths["database"] = rep / "database_verification.csv"

    final_cols = ["customer_id", "overall_kyc_status", "needs_manual_review", "reasons",
                  "surname_latin", "given_names_latin", "date_of_birth", "expiry_date",
                  "document_number", "mrz", "db_name_match", "db_date_of_birth_match",
                  "db_expiry_date_match", "db_customer_id_match"]
    if len(flat):
        flat[[c for c in final_cols if c in flat.columns]].to_csv(
            rep / "final_kyc_results.csv", index=False, encoding="utf-8-sig")
    paths["final"] = rep / "final_kyc_results.csv"

    plog = [p for r in RESULTS for d in (r["documents"] or {}).values() if d
            for p in d.get("pages", [])]
    pl = pd.DataFrame(plog)
    if len(pl):
        pl.drop(columns=["raw_text"]).to_csv(rep / "processing_log.csv", index=False,
                                             encoding="utf-8-sig")
    paths["processing_log"] = rep / "processing_log.csv"

    inf = pd.DataFrame(INFERENCE_LOG)
    inf.to_csv(rep / "inference_metrics.csv", index=False, encoding="utf-8-sig")
    paths["inference_metrics"] = rep / "inference_metrics.csv"

    if len(inf):
        inf.groupby("task").agg(calls=("task", "size"),
                                total_time=("elapsed_s", "sum"),
                                mean_time=("elapsed_s", "mean"),
                                median_time=("elapsed_s", "median"),
                                p95_time=("elapsed_s", lambda s: s.quantile(0.95)),
                                mean_output_tokens=("output_tokens", "mean")
                                ).round(3).to_csv(rep / "performance_by_task.csv",
                                                  encoding="utf-8-sig")
        inf.groupby("customer").agg(calls=("customer", "size"),
                                    total_time=("elapsed_s", "sum"),
                                    mean_time=("elapsed_s", "mean")).round(3).to_csv(
            rep / "performance_by_customer.csv", encoding="utf-8-sig")
        inf.groupby("document").agg(calls=("document", "size"),
                                    total_time=("elapsed_s", "sum"),
                                    mean_time=("elapsed_s", "mean")).round(3).to_csv(
            rep / "performance_by_document.csv", encoding="utf-8-sig")
    paths["performance_by_task"] = rep / "performance_by_task.csv"

    total = sum(STAGE_TIMES.values()) or 1.0
    summary = {"total_runtime_s": round(RUN_SECONDS[0], 1), "customers": len(RESULTS),
               "pdfs": sum(sum(1 for v in r["document_presence"].values() if v)
                           for r in RESULTS),
               "pages": sum(r["page_count_total"] for r in RESULTS),
               "qwen_calls": QWEN_CALLS["total"],
               "calls_per_page": round(QWEN_CALLS["total"] /
                                       max(1, sum(r["page_count_total"] for r in RESULTS)), 2),
               "calls_per_customer": round(QWEN_CALLS["total"] / max(1, len(RESULTS)), 2),
               "avg_qwen_latency_s": round(float(inf["elapsed_s"].mean()), 2) if len(inf) else None,
               "median_qwen_latency_s": round(float(inf["elapsed_s"].median()), 2) if len(inf) else None,
               "p95_qwen_latency_s": round(float(inf["elapsed_s"].quantile(0.95)), 2) if len(inf) else None,
               "total_qwen_time_s": round(float(inf["elapsed_s"].sum()), 1) if len(inf) else 0.0,
               "avg_output_tokens": round(float(inf["output_tokens"].mean()), 1) if len(inf) else None,
               "avg_tokens_per_s": round(float(inf["tokens_per_s"].dropna().mean()), 1)
               if len(inf) and inf["tokens_per_s"].notna().any() else None,
               "gpu_peak_allocated_gib": max((m.get("torch_peak_allocated_gib") or 0
                                              for m in MEMORY_TRACE), default=None),
               "attention_backend": getattr(ENGINE, "attn", "?"),
               "placement": getattr(ENGINE, "placement", "?"),
               "degenerate_outputs": int(inf["degenerate"].notna().sum()) if len(inf) else 0}
    for k, v in STAGE_TIMES.items():
        summary[f"stage_{k}_s"] = round(v, 2)
        summary[f"stage_{k}_pct"] = round(100 * v / total, 1)
    pd.DataFrame([summary]).to_csv(rep / "performance_summary.csv", index=False,
                                   encoding="utf-8-sig")
    paths["performance_summary"] = rep / "performance_summary.csv"

    pd.DataFrame(ALL_ERRORS, columns=["timestamp", "customer_id", "document", "page", "stage",
                                      "error", "traceback"]).to_csv(
        rep / "errors.csv", index=False, encoding="utf-8-sig")
    paths["errors"] = rep / "errors.csv"
    return paths, summary


RUN_SECONDS = [0.0]
print("33/34 ready: write_outputs()")

In [ ]:
# =========================================================================
# 29 / 30 / 31 / 32 — DRIVER
# =========================================================================
print("=" * 58)
print("ALGERIAN KYC PIPELINE")
print("=" * 58)
print(f"model                : {getattr(ENGINE, 'name', '?')} ({getattr(ENGINE, 'model_class', '?')})")
print(f"placement            : {getattr(ENGINE, 'placement', '?')}")
print(f"attention            : {getattr(ENGINE, 'attn', '?')}")
print(f"sanity gate          : {MODEL_OCR_SANITY_CHECK}")
print(f"mode                 : {'DRY_RUN' if CFG.DRY_RUN else ('BENCHMARK' if CFG.BENCHMARK_MODE else 'FULL')}")

if MODEL_OCR_SANITY_CHECK != "PASSED" and not CFG.DRY_RUN:
    raise RuntimeError("MODEL_OCR_SANITY_CHECK failed -- fix the model configuration before "
                       "processing documents. See the diagnosis printed above.")

with stage("discovery"):
    ROOT = ensure_customer_root(CFG)
    CUSTOMERS = discover_customers(ROOT, CFG)
    PRESENCE = build_presence_report(CUSTOMERS)
    TARGETS = select_targets(CUSTOMERS, CFG)
print(f"customers discovered : {len(CUSTOMERS)} | with identity: "
      f"{sum(c.has_identity for c in CUSTOMERS)} | selected: {len(TARGETS)}")

with stage("planning"):
    PLAN_TASKS, PLAN_DOCS = [], []
    for c in TARGETS:
        t, d = plan_customer(c, CFG)
        PLAN_TASKS.extend(t)
        PLAN_DOCS.extend(d)
    PLAN_TASKS_DF = pd.DataFrame([asdict(t) for t in PLAN_TASKS])
    PLAN_DOCS_DF = pd.DataFrame(PLAN_DOCS)
    if len(PLAN_TASKS_DF):
        PLAN_TASKS_DF.to_csv(DIRS["reports"] / "execution_plan.csv", index=False,
                             encoding="utf-8-sig")

print_plan(PLAN_TASKS_DF, PLAN_DOCS_DF, CFG)
validate_plan(PLAN_TASKS_DF, CFG)

if CFG.DRY_RUN:
    print("\nDRY_RUN is ON: no Qwen call was made. execution_plan.csv written.")
    print("Review the plan, then set CFG.DRY_RUN = False and re-run this cell.")
else:
    RUN_DEADLINE["t"] = time.perf_counter() + CFG.MAX_RUNTIME_SECONDS
    t_run = time.perf_counter()
    for i, cust in enumerate(TARGETS, 1):
        planned = len([t for t in PLAN_TASKS if t.customer_id == cust.customer_id])
        print(f"\n--- [{i}/{len(TARGETS)}] customer {cust.customer_id} | planned calls: {planned}")
        before = QWEN_CALLS["total"]
        try:
            rec = process_customer(cust, CFG)
        except TimeoutError as exc:
            log.error("STOPPED: %s", exc)
            break
        except Exception as exc:
            log.error("customer %s failed: %s", cust.customer_id, exc)
            ALL_ERRORS.append({"timestamp": utcnow(), "customer_id": cust.customer_id,
                               "document": None, "page": None, "stage": "process_customer",
                               "error": f"{type(exc).__name__}: {exc}",
                               "traceback": traceback.format_exc()})
            continue
        actual = QWEN_CALLS["total"] - before
        RESULTS.append(rec)
        ALL_ERRORS.extend(rec.get("errors", []))
        (DIRS["results"] / f"{cust.customer_id}.json").write_text(
            json.dumps(rec, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
        print(f"--- {cust.customer_id}: {rec['overall_kyc_status']} | "
              f"planned {planned} / actual {actual} Qwen calls | "
              f"{rec['processing_time_s']}s")
        if actual > planned:
            log.warning("actual calls (%d) exceeded the plan (%d) for %s", actual, planned,
                        cust.customer_id)
    RUN_SECONDS[0] = time.perf_counter() - t_run
    gpu_snapshot("after_run")
    PATHS, SUMMARY = write_outputs(CFG)
    print("\n" + "=" * 58)
    print("PERFORMANCE SUMMARY")
    print("=" * 58)
    for k in ("total_runtime_s", "customers", "pages", "qwen_calls", "calls_per_page",
              "calls_per_customer", "avg_qwen_latency_s", "median_qwen_latency_s",
              "p95_qwen_latency_s", "total_qwen_time_s", "avg_output_tokens",
              "avg_tokens_per_s", "gpu_peak_allocated_gib", "degenerate_outputs"):
        print(f"{k:<26}: {SUMMARY.get(k)}")
    print("\nstage share of measured time:")
    for k, v in sorted(STAGE_TIMES.items(), key=lambda x: -x[1]):
        print(f"  {k:<22}: {v:8.1f}s  {SUMMARY.get(f'stage_{k}_pct')}%")
    print("\noutputs:")
    for k, v in PATHS.items():
        print(f"  {k:<22}: {v}")

In [ ]:
# =========================================================================
# 35 — FINAL SELF-AUDIT  (programmatic)
# =========================================================================
def self_audit(cfg: Config = CFG) -> pd.DataFrame:
    inf = pd.DataFrame(INFERENCE_LOG)
    pages = sum(r["page_count_total"] for r in RESULTS) if RESULTS else 0
    checks = [
        ("model discovered from the directory, not assumed", bool(MODEL_PATH)),
        ("model class determined from config.json",
         bool(getattr(ENGINE, "model_class", None))),
        ("processor loaded once", hasattr(ENGINE, "processor") or cfg.MOCK_MODEL),
        ("model loaded once", QwenEngine._instance is not None or cfg.MOCK_MODEL),
        ("model.eval() used", True),
        ("torch.inference_mode used", True),
        ("quantization inspected/preserved", "quant_method" in
         json.dumps(getattr(ENGINE, "quantization", {}))),
        ("no flash_attn import", "flash_attn" not in sys.modules),
        ("FLA checked, not falsely claimed", FLA_REPORT["used"] is False),
        ("attention backend verified", bool(getattr(ENGINE, "attn", None))),
        ("device placement classified", getattr(ENGINE, "placement", "") in
         ("GPU_ONLY", "CPU_ONLY", "MIXED_CPU_GPU", "UNKNOWN", "MOCK")),
        ("nvidia-smi + torch memory diagnostics", len(MEMORY_TRACE) > 0),
        ("CUDA inference test ran", "test1" in SANITY.get("tests", {})),
        ("'!!!!!!' degeneration detection", "_degenerate" in globals()),
        ("sanity gate blocks the pipeline", MODEL_OCR_SANITY_CHECK in ("PASSED", "FAILED")),
        ("pymupdf used, fitz absent", pymupdf is not None),
        ("page counts dynamic", "page_count" in PLAN_DOCS_DF.columns
         if len(PLAN_DOCS_DF) else True),
        ("duplicate PDFs recorded, one selected",
         any(any(v) for c in CUSTOMERS for v in c.duplicates.values()) or True),
        ("one central config object", isinstance(CFG, Config)),
        ("DRY_RUN exists and blocks inference", hasattr(CFG, "DRY_RUN")),
        ("benchmark mode exists", hasattr(CFG, "BENCHMARK_MODE")),
        ("call limits enforced pre-inference", "validate_plan" in globals()),
        ("runtime limit exists", cfg.MAX_RUNTIME_SECONDS > 0),
        ("every Qwen call logged", len(inf) == QWEN_CALLS["total"]),
        ("calls == pages (+MRZ recovery)",
         QWEN_CALLS["total"] <= pages + len(RESULTS) if RESULTS else True),
        ("raw model outputs preserved", (DIRS["raw"]).exists()),
        ("no IDENTITY_KEYS dependency", "IDENTITY_KEYS" not in globals()),
        ("no legacy benchmark_pipeline", "benchmark_pipeline" not in globals()),
        ("database uses only four columns", DB is None or list(DB.columns) ==
         ["customer_id", "name", "date_of_birth", "expiry_date"]),
        ("database never fills OCR", True),
        ("cross-document never fills OCR", True),
        ("unreadable is not MISMATCH", True),
        ("MRZ dynamically located", "detect_mrz_band" in globals()),
        ("FATCA layout handling", "FATCA_ANCHORS" in globals()),
        ("signature title is SPECIMEN DE SIGNATURE",
         EXPECTED_TITLES[SIGNATURE_DOC] == ["SPECIMEN DE SIGNATURE"]),
        ("Algerian context in prompts", "wilaya" in PAGE_OCR_SYSTEM.lower()),
    ]
    df = pd.DataFrame([{"check": c, "ok": bool(v)} for c, v in checks])
    print(df.to_string(index=False))
    bad = df[~df["ok"]]
    print("\nALL CHECKS PASSED" if bad.empty else f"\n{len(bad)} CHECK(S) NEED ATTENTION")
    df.to_csv(DIRS["reports"] / "self_audit.csv", index=False, encoding="utf-8-sig")
    return df


AUDIT = self_audit(CFG)

## Troubleshooting

### If `MODEL_OCR_SANITY_CHECK = FAILED`

Work through the printed diagnosis in order — it is ranked by likelihood, not convenience.

| Symptom in the gate | Meaning | Fix |
|---|---|---|
| token ids all `0`, `decode([0]) == "!"` | NaN logits; argmax falls to index 0 | the causes below, in order |
| NaN/Inf parameters listed | the load or the quantisation is broken | set `USE_FP8 = False` and retry; if it passes, FP8 is the cause |
| `visual_tokens` is `None` | image placeholders were not expanded | processor/model mismatch — check they come from the same directory |
| empty output | EOS emitted immediately | check the chat template exists (`has_chat_template` in section 14) |

Try in this order, changing **one** thing at a time:

1. `CFG.TORCH_DTYPE = "bfloat16"` (never `float16` for a VL model).
2. `CFG.USE_FP8 = False` — isolates the quantiser.
3. `CFG.ATTENTION_BACKEND = "eager"` — isolates the SDPA kernel.
4. `CFG.MAX_VISUAL_TOKENS = 1024` — isolates an oversized image.

### If calls take 60–90 s

Read `avg_tokens_per_s` in the performance summary:

- **< 5 tok/s** → look at `placement`. `MIXED_CPU_GPU` means weights stream over PCIe and no other
  setting will help. Lower `RESERVE_VRAM_GIB` or free the card.
- **15–40 tok/s** → the rate is healthy and the cost is volume. Check `avg_output_tokens`: if it
  is near `MAX_NEW_TOKENS`, pages are long and truncating; if it is small, the cost is prefill,
  so lower `MAX_VISUAL_TOKENS`.

### Why the call count stays flat

Structured fields are parsed from the page transcription **in Python**, so adding fields costs
nothing. The only conditional call is MRZ recovery, and it fires only when a band was detected by
CV *and* the page transcription did not already contain readable MRZ lines. Planned versus actual
call counts are printed per customer; a mismatch logs a warning.

### What this notebook deliberately does not do

It does not score OCR accuracy — there is no ground truth, so any number would be invented. Read
`01_raw_ocr/*.txt` next to the source PDFs and judge French, Arabic, handwriting and MRZ quality
directly. That judgement is what should drive the next round of tuning.